## 0. Setup 


In [ ]:
import os, sys, subprocess, zipfile

IN_KAGGLE = os.path.isdir('/kaggle/input')

def _find_dir(roots, must_contain):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, _ in os.walk(root):
            if all(os.path.exists(os.path.join(dirpath, m)) for m in must_contain):
                return dirpath
    return None

def _find_file(roots, filename):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None

def _extract_zips(roots, dest):
    os.makedirs(dest, exist_ok=True)
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if os.path.abspath(dirpath).startswith(os.path.abspath(dest)):
                continue
            for fn in filenames:
                if not fn.lower().endswith('.zip'):
                    continue
                marker = os.path.join(dest, '.' + fn + '.done')
                if os.path.exists(marker):
                    continue
                src = os.path.join(dirpath, fn)
                try:
                    with zipfile.ZipFile(src) as zf:
                        zf.extractall(dest)
                    open(marker, 'w').close()
                    print('extracted:', src)
                except zipfile.BadZipFile:
                    print('skip (not a zip):', src)

WEIGHT_NAME = '03_ResNet34_Transformer_OCR.pth'   # baseline A (eval-only)

if IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'albumentations'], check=False)
    INPUT_ROOTS = ['/kaggle/input']
    EXTRACT_DIR = '/kaggle/working/_data'
    SRC_DIR     = '/kaggle/working/src'
    OUT_DIR     = '/kaggle/working'
else:
    INPUT_ROOTS = ['..']
    EXTRACT_DIR = os.path.abspath('../_data')
    SRC_DIR     = os.path.abspath('./_src')
    OUT_DIR     = '..'

DATA_ROOT   = _find_dir(INPUT_ROOTS, ['train', 'test_label'])
WEIGHT_PATH = _find_file(INPUT_ROOTS, WEIGHT_NAME)
if not (DATA_ROOT and WEIGHT_PATH):
    print('Some inputs not found extracted -> unzipping archives ...')
    _extract_zips(INPUT_ROOTS, EXTRACT_DIR)
    SEARCH = INPUT_ROOTS + [EXTRACT_DIR]
    DATA_ROOT   = DATA_ROOT   or _find_dir(SEARCH, ['train', 'test_label'])
    WEIGHT_PATH = WEIGHT_PATH or _find_file(SEARCH, WEIGHT_NAME)

assert DATA_ROOT, 'Could not find data (needs train/ and test_label/) in inputs or zips.'
assert WEIGHT_PATH, ('Baseline weight ' + WEIGHT_NAME + ' not found in inputs or zips. '
                     'Can weight 03 lam baseline A.')

print('IN_KAGGLE :', IN_KAGGLE)
print('DATA_ROOT :', DATA_ROOT)
print('WEIGHT(A) :', WEIGHT_PATH)
print('OUT_DIR   :', OUT_DIR)
print('SRC_DIR   :', SRC_DIR)

## 1. Ghi code (nhúng sẵn) ra đĩa rồi thêm vào sys.path

In [ ]:
import base64, os, sys

_FILES = {
"config/__init__.py": "ZnJvbSAuY29uZmlnIGltcG9ydCBDb25maWcNCmZyb20gLnByZXNldHMgaW1wb3J0IG1ha2VfY29uZmlnLCBQUkVTRVRTLCBCQVNFTElORVMsIEZVU0lPTlMsIFNSLCBTSU5HTEVfRlJBTUUNCg0KX19hbGxfXyA9IFsnQ29uZmlnJywgJ21ha2VfY29uZmlnJywgJ1BSRVNFVFMnLCAnQkFTRUxJTkVTJywgJ0ZVU0lPTlMnLCAnU1InLCAnU0lOR0xFX0ZSQU1FJ10NCg==",
"config/config.py": "ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZA0KZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIFR1cGxlDQppbXBvcnQgdG9yY2gNCg0KDQpAZGF0YWNsYXNzDQpjbGFzcyBDb25maWc6DQogICAgcGF0aDogc3RyID0gJy4uL2RhdGEnDQogICAgc2VlZDogaW50ID0gNDINCiAgICBkZXZpY2U6IHN0ciA9ICdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScNCg0KICAgICMgRGF0YXNldCBjb25maWcNCiAgICB2b2NhYjogc3RyID0gIjAxMjM0NTY3ODlBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWiINCiAgICBudW1fY2xhc3NlczogaW50ID0gZmllbGQoaW5pdD1GYWxzZSkNCiAgICBsYWJlbF9sZW46IGludCA9IDcNCiAgICBpbWdfSDogaW50ID0gMzINCiAgICBpbWdfVzogaW50ID0gMTI4DQogICAgbnVtX2ZyYW1lczogaW50ID0gNQ0KICAgIGxheW91dHM6IFR1cGxlW3N0ciwgLi4uXSA9ICgnQnJhemlsaWFuJywgJ01lcmNvc3VyJykNCg0KICAgICMgTW9kZWwgY29uZmlnDQogICAgZW1iZWRfZGltOiBpbnQgPSA1MTINCiAgICBmZl9kaW06IGludCA9IDUxMiAqIDQNCiAgICBudW1fbGF5ZXJzOiBpbnQgPSAzDQogICAgbnVtX2hlYWRzOiBpbnQgPSA4DQogICAgZHJvcF9vdXQ6IGZsb2F0ID0gMC4xDQoNCiAgICAjIFJlZ2lzdHJ5IGNob2ljZXMgKGNvbGxhcHNlIHRoZSA3IGJhc2VsaW5lIG5vdGVib29rcyBpbnRvIG9uZSBtb2RlbCkNCiAgICBiYWNrYm9uZTogc3RyID0gJ3Jlc25ldDUwJyAgICAgICAgICAjIHJlc25ldDM0IHwgcmVzbmV0NTAgfCBjb252bmV4dF90aW55IHwgY29udm5leHRfYmFzZQ0KICAgIGRlY29kZXI6IHN0ciA9ICd0cmFuc2Zvcm1lcicgICAgICAgICMgdHJhbnNmb3JtZXIgfCBiaWxzdG0NCiAgICBmdXNpb25fdHlwZTogc3RyID0gJ2F0dGVudGlvbicgICAgICAjIG1lYW4gfCBtYXggfCBhdHRlbnRpb24gfCBmcmFtZV9xdWFsaXR5IHwgdGVtcG9yYWxfdHJhbnNmb3JtZXINCiAgICBtdWx0aV9mcmFtZTogYm9vbCA9IFRydWUgICAgICAgICAgICAjIEZhbHNlIC0+IHNpbmdsZS1mcmFtZSBiYXNlbGluZQ0KICAgIGV4dHJhY3Rvcl9wcmV0cmFpbmVkOiBib29sID0gVHJ1ZQ0KICAgIGZyZWV6ZV9leHRyYWN0b3I6IGJvb2wgPSBUcnVlDQoNCiAgICAjIFN1cGVyLXJlc29sdXRpb24gKG11bHRpLXRhc2spIGJyYW5jaA0KICAgIHVzZV9zcjogYm9vbCA9IEZhbHNlDQogICAgc3Jfc2NhbGU6IGludCA9IDINCiAgICBzcl9sb3NzX3dlaWdodDogZmxvYXQgPSAwLjENCg0KICAgICMgTGF5b3V0LWNsYXNzaWZpY2F0aW9uIGhlYWQgKGZvciBsYXlvdXQtYXdhcmUgcG9zdC1wcm9jZXNzaW5nKQ0KICAgIHVzZV9sYXlvdXRfaGVhZDogYm9vbCA9IEZhbHNlDQogICAgbGF5b3V0X2xvc3Nfd2VpZ2h0OiBmbG9hdCA9IDAuNQ0KDQogICAgIyBUcmFpbmluZyBjb25maWcNCiAgICBscjogZmxvYXQgPSA1ZS00DQogICAgYmF0Y2hfc2l6ZTogaW50ID0gNjQNCiAgICBlcG9jaHM6IGludCA9IDMwDQogICAgbG9nX2ludGVydmFsOiBpbnQgPSAxDQogICAgZWFybHlfc3RvcF9jb3VudDogaW50ID0gNQ0KICAgIHdhcm11cF9lcG9jaHM6IGludCA9IDMNCiAgICB1c2VfYW1wOiBib29sID0gRmFsc2UgICAgICAgICAgICAgICAjIG1peGVkIHByZWNpc2lvbiAoQ1VEQSBvbmx5KTsgbm8tb3Agb24gQ1BVDQogICAgdHJhaW5fbG9zczogTGlzdFtmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkNCiAgICB2YWxfbG9zczogTGlzdFtmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkNCiAgICB0cmFpbl9hY2M6IExpc3RbZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpDQogICAgdmFsX2FjYzogTGlzdFtmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkNCiAgICBiZXN0X21vZGVsX3BhdGg6IHN0ciA9ICdSZXNUcmFuT0NSLnB0aCcNCg0KICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpOg0KICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gbGVuKHNlbGYudm9jYWIpDQo=",
"config/presets.py": "IiIiTmFtZWQgZXhwZXJpbWVudCBwcmVzZXRzIOKAlCB0aGUgc2luZ2xlIHNvdXJjZSBvZiB0cnV0aCB0aGF0IHJlcGxhY2VzIHRoZQo3IG5lYXItaWRlbnRpY2FsIGJhc2VsaW5lIG5vdGVib29rcy4KCkVhY2ggcHJlc2V0IGlzIGp1c3QgYSA6Y2xhc3M6YENvbmZpZ2Agd2l0aCB0aGUgcmVsZXZhbnQgcmVnaXN0cnkgZmllbGRzIHNldCwgc28KcnVubmluZyBhbiBleHBlcmltZW50IGlzICJwaWNrIGEgbmFtZSIgaW5zdGVhZCBvZiAiZWRpdC9jb3B5IGEgbm90ZWJvb2siLgoKICAgIGZyb20gY29uZmlnIGltcG9ydCBDb25maWcKICAgIGZyb20gY29uZmlnLnByZXNldHMgaW1wb3J0IG1ha2VfY29uZmlnLCBCQVNFTElORVMsIEZVU0lPTlMKCiAgICBjZmcgPSBtYWtlX2NvbmZpZygicmVzbmV0MzRfdHJhbnNmb3JtZXIiKSAgICAgICAgICAjIG9uZSBiYXNlbGluZQogICAgY2ZnID0gbWFrZV9jb25maWcoInJlc25ldDM0X3RyYW5zZm9ybWVyIiwgdXNlX3NyPVRydWUsIGZ1c2lvbl90eXBlPSJtZWFuIikKIiIiCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCByZXBsYWNlCgpmcm9tIC5jb25maWcgaW1wb3J0IENvbmZpZwoKCmRlZiBtYWtlX2NvbmZpZyhwcmVzZXQ9InJlc25ldDUwX3RyYW5zZm9ybWVyIiwgKipvdmVycmlkZXMpOgogICAgIiIiQnVpbGQgYSBDb25maWcgZnJvbSBhIHByZXNldCBuYW1lIHBsdXMgb3B0aW9uYWwgZmllbGQgb3ZlcnJpZGVzLiIiIgogICAgaWYgcHJlc2V0IG5vdCBpbiBQUkVTRVRTOgogICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBwcmVzZXQgJ3twcmVzZXR9Jy4gQXZhaWxhYmxlOiB7c29ydGVkKFBSRVNFVFMpfSIpCiAgICBiYXNlID0gUFJFU0VUU1twcmVzZXRdKCkKICAgIGlmIG92ZXJyaWRlczoKICAgICAgICBiYXNlID0gcmVwbGFjZShiYXNlLCAqKm92ZXJyaWRlcykKICAgIHJldHVybiBiYXNlCgoKZGVmIF9jZmcoYmFja2JvbmUsIGRlY29kZXIsIGZ1c2lvbl90eXBlPSJhdHRlbnRpb24iLCBtdWx0aV9mcmFtZT1UcnVlLCAqKmt3KToKICAgIHJldHVybiBsYW1iZGE6IENvbmZpZygKICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwgZGVjb2Rlcj1kZWNvZGVyLCBmdXNpb25fdHlwZT1mdXNpb25fdHlwZSwKICAgICAgICBtdWx0aV9mcmFtZT1tdWx0aV9mcmFtZSwgKiprdywKICAgICkKCgojIOKUgOKUgCBCYXNlbGluZXM6IDQgYmFja2JvbmVzIHggMiBkZWNvZGVycyAoZnVzaW9uPWF0dGVudGlvbikg4oCUIHJlcGxhY2VzIG5iIDAyLi4wOCDilIDilIAKQkFTRUxJTkVTID0gewogICAgInJlc25ldDUwX3RyYW5zZm9ybWVyIjogX2NmZygicmVzbmV0NTAiLCAidHJhbnNmb3JtZXIiKSwKICAgICJyZXNuZXQ1MF9iaWxzdG0iOiBfY2ZnKCJyZXNuZXQ1MCIsICJiaWxzdG0iKSwKICAgICJyZXNuZXQzNF90cmFuc2Zvcm1lciI6IF9jZmcoInJlc25ldDM0IiwgInRyYW5zZm9ybWVyIiksCiAgICAicmVzbmV0MzRfYmlsc3RtIjogX2NmZygicmVzbmV0MzQiLCAiYmlsc3RtIiksCiAgICAiY29udm5leHRfdGlueV90cmFuc2Zvcm1lciI6IF9jZmcoImNvbnZuZXh0X3RpbnkiLCAidHJhbnNmb3JtZXIiKSwKICAgICJjb252bmV4dF90aW55X2JpbHN0bSI6IF9jZmcoImNvbnZuZXh0X3RpbnkiLCAiYmlsc3RtIiksCiAgICAiY29udm5leHRfYmFzZV90cmFuc2Zvcm1lciI6IF9jZmcoImNvbnZuZXh0X2Jhc2UiLCAidHJhbnNmb3JtZXIiKSwKICAgICJjb252bmV4dF9iYXNlX2JpbHN0bSI6IF9jZmcoImNvbnZuZXh0X2Jhc2UiLCAiYmlsc3RtIiksCn0KCiMg4pSA4pSAIFNpbmdsZS1mcmFtZSBiYXNlbGluZSAoYWJsYXRpb246IG5vIG11bHRpLWZyYW1lIGZ1c2lvbikg4pSA4pSAClNJTkdMRV9GUkFNRSA9IHsKICAgICJzaW5nbGVfZnJhbWVfcmVzbmV0MzQiOiBfY2ZnKCJyZXNuZXQzNCIsICJ0cmFuc2Zvcm1lciIsIG11bHRpX2ZyYW1lPUZhbHNlKSwKfQoKIyDilIDilIAgRnVzaW9uIHN3ZWVwIG9uIGEgZml4ZWQgYmFja2JvbmUvZGVjb2RlciDilIDilIAKRlVTSU9OUyA9IHsKICAgIGYiZnVzaW9uX3tmfSI6IF9jZmcoInJlc25ldDM0IiwgInRyYW5zZm9ybWVyIiwgZnVzaW9uX3R5cGU9ZikKICAgIGZvciBmIGluIFsibWVhbiIsICJtYXgiLCAiYXR0ZW50aW9uIiwgImZyYW1lX3F1YWxpdHkiLCAidGVtcG9yYWxfdHJhbnNmb3JtZXIiXQp9CgojIOKUgOKUgCBTdXBlci1yZXNvbHV0aW9uIG11bHRpLXRhc2sgdmFyaWFudHMg4pSA4pSAClNSID0gewogICAgInNyX3Jlc25ldDM0X3RyYW5zZm9ybWVyIjogX2NmZygicmVzbmV0MzQiLCAidHJhbnNmb3JtZXIiLCB1c2Vfc3I9VHJ1ZSksCiAgICAic3JfbGF5b3V0X3Jlc25ldDM0X3RyYW5zZm9ybWVyIjogX2NmZygicmVzbmV0MzQiLCAidHJhbnNmb3JtZXIiLCB1c2Vfc3I9VHJ1ZSwgdXNlX2xheW91dF9oZWFkPVRydWUpLAp9CgpQUkVTRVRTID0geyoqQkFTRUxJTkVTLCAqKlNJTkdMRV9GUkFNRSwgKipGVVNJT05TLCAqKlNSfQo=",
"models/__init__.py": "ZnJvbSAuY29tcG9uZW50cyBpbXBvcnQgKA0KICAgIEF0dGVudGlvbkZ1c2lvbiwNCiAgICBGZWF0dXJlRXh0cmFjdG9yLA0KICAgIFNUTkJsb2NrLA0KICAgIFRyYW5zZm9ybWVyRW5jb2RlciwNCiAgICBUcmFuc2Zvcm1lckVuY29kZXJCbG9jaywNCiAgICBwb3NfZW5jb2RpbmcsDQopDQpmcm9tIC5iYWNrYm9uZXMgaW1wb3J0IGJ1aWxkX2JhY2tib25lDQpmcm9tIC5mdXNpb24gaW1wb3J0IGJ1aWxkX2Z1c2lvbg0KZnJvbSAuZGVjb2RlcnMgaW1wb3J0IGJ1aWxkX2RlY29kZXINCmZyb20gLnNyIGltcG9ydCBTUk1vZHVsZQ0KZnJvbSAucmVzdHJhbnNPUkMgaW1wb3J0IFJlc1RyYW5PQ1IsIGJ1aWxkX21vZGVsLCBMYXlvdXRIZWFkDQoNCl9fYWxsX18gPSBbDQogICAgJ0F0dGVudGlvbkZ1c2lvbicsDQogICAgJ0ZlYXR1cmVFeHRyYWN0b3InLA0KICAgICdTVE5CbG9jaycsDQogICAgJ1RyYW5zZm9ybWVyRW5jb2RlcicsDQogICAgJ1RyYW5zZm9ybWVyRW5jb2RlckJsb2NrJywNCiAgICAncG9zX2VuY29kaW5nJywNCiAgICAnUmVzVHJhbk9DUicsDQogICAgJ2J1aWxkX21vZGVsJywNCiAgICAnYnVpbGRfYmFja2JvbmUnLA0KICAgICdidWlsZF9mdXNpb24nLA0KICAgICdidWlsZF9kZWNvZGVyJywNCiAgICAnU1JNb2R1bGUnLA0KICAgICdMYXlvdXRIZWFkJywNCl0NCg==",
"models/backbones.py": "IiIiQmFja2JvbmUgcmVnaXN0cnkg4oCUIGZlYXR1cmUgZXh0cmFjdG9ycyB0aGF0IG91dHB1dCAoQiwgb3V0X2RpbSwgMSwgVycpLgoKQWxsIGJhY2tib25lcyBtb2RpZnkgdGhlaXIgc3RyaWRlIHNvIHRoZSBoZWlnaHQgY29sbGFwc2VzIHdoaWxlIHRoZSB3aWR0aAood2hpY2ggYmVjb21lcyB0aGUgT0NSIHNlcXVlbmNlIGF4aXMpIGlzIHByZXNlcnZlZC4gVGhlIG91dHB1dCBpcyBoZWlnaHQtcG9vbGVkCnRvIDEgYW5kIHByb2plY3RlZCB0byBgYG91dF9kaW1gYCBzbyBldmVyeSBiYWNrYm9uZSBpcyBpbnRlcmNoYW5nZWFibGUuCgpLZWVwaW5nIHRoZSBzdWJtb2R1bGUgYXR0cmlidXRlIG5hbWVzIChgYGNvbnYxYGAvYGBsYXllcjFgYC9gYHByb2pgYCBmb3IgUmVzTmV0LApgYGZlYXR1cmVzYGAvYGBwcm9qYGAgZm9yIENvbnZOZVh0KSBpZGVudGljYWwgdG8gdGhlIG9yaWdpbmFsIG5vdGVib29rcyBtZWFucwpjaGVja3BvaW50cyB0cmFpbmVkIHRoZXJlIHN0aWxsIGxvYWQgaW50byB0aGlzIHJlZ2lzdHJ5LgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRvcmNodmlzaW9uLm1vZGVscyBpbXBvcnQgKAogICAgcmVzbmV0MzQsIFJlc05ldDM0X1dlaWdodHMsCiAgICByZXNuZXQ1MCwgUmVzTmV0NTBfV2VpZ2h0cywKICAgIGNvbnZuZXh0X3RpbnksIENvbnZOZVh0X1RpbnlfV2VpZ2h0cywKICAgIGNvbnZuZXh0X2Jhc2UsIENvbnZOZVh0X0Jhc2VfV2VpZ2h0cywKKQoKCmNsYXNzIFJlc05ldEV4dHJhY3Rvcihubi5Nb2R1bGUpOgogICAgIiIiUmVzTmV0MzQvNTAgYmFja2JvbmUgd2l0aCBoZWlnaHQtY29sbGFwc2luZyBzdHJpZGUgbW9kaWZpY2F0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2YXJpYW50PSJyZXNuZXQ1MCIsIHByZXRyYWluZWQ9VHJ1ZSwgb3V0X2RpbT01MTIsIGZyZWV6ZV9iYWNrYm9uZT1GYWxzZSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgdmFyaWFudCA9PSAicmVzbmV0NTAiOgogICAgICAgICAgICBiYWNrYm9uZSA9IHJlc25ldDUwKHdlaWdodHM9UmVzTmV0NTBfV2VpZ2h0cy5JTUFHRU5FVDFLX1YyIGlmIHByZXRyYWluZWQgZWxzZSBOb25lKQogICAgICAgICAgICBmaW5hbF9jaGFubmVscyA9IDIwNDgKICAgICAgICAgICAgaXNfYm90dGxlbmVjayA9IFRydWUKICAgICAgICBlbGlmIHZhcmlhbnQgPT0gInJlc25ldDM0IjoKICAgICAgICAgICAgYmFja2JvbmUgPSByZXNuZXQzNCh3ZWlnaHRzPVJlc05ldDM0X1dlaWdodHMuSU1BR0VORVQxS19WMSBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZSkKICAgICAgICAgICAgZmluYWxfY2hhbm5lbHMgPSA1MTIKICAgICAgICAgICAgaXNfYm90dGxlbmVjayA9IEZhbHNlCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gcmVzbmV0IHZhcmlhbnQ6IHt2YXJpYW50fSIpCgogICAgICAgIHNlbGYuY29udjEgPSBiYWNrYm9uZS5jb252MQogICAgICAgIHNlbGYuYm4xID0gYmFja2JvbmUuYm4xCiAgICAgICAgc2VsZi5yZWx1ID0gYmFja2JvbmUucmVsdQogICAgICAgIHNlbGYubWF4cG9vbCA9IGJhY2tib25lLm1heHBvb2wKICAgICAgICBzZWxmLmxheWVyMSA9IGJhY2tib25lLmxheWVyMQogICAgICAgIHNlbGYubGF5ZXIyID0gYmFja2JvbmUubGF5ZXIyCiAgICAgICAgc2VsZi5sYXllcjMgPSBiYWNrYm9uZS5sYXllcjMKICAgICAgICBzZWxmLmxheWVyNCA9IGJhY2tib25lLmxheWVyNAoKICAgICAgICAjIE1vZGlmeSBzdHJpZGUgKDIsMikgLT4gKDIsMSk6IGtlZXAgd2lkdGgsIHNocmluayBoZWlnaHQgb25seS4KICAgICAgICAjIEJvdHRsZW5lY2sgKFJlc05ldDUwKSBjYXJyaWVzIHRoZSBzdHJpZGUgb24gY29udjI7IEJhc2ljQmxvY2sgKFJlc05ldDM0KSBvbiBjb252MS4KICAgICAgICBzdHJpZGVfY29udiA9ICJjb252MiIgaWYgaXNfYm90dGxlbmVjayBlbHNlICJjb252MSIKICAgICAgICBnZXRhdHRyKHNlbGYubGF5ZXIzWzBdLCBzdHJpZGVfY29udikuc3RyaWRlID0gKDIsIDEpCiAgICAgICAgc2VsZi5sYXllcjNbMF0uZG93bnNhbXBsZVswXS5zdHJpZGUgPSAoMiwgMSkKICAgICAgICBnZXRhdHRyKHNlbGYubGF5ZXI0WzBdLCBzdHJpZGVfY29udikuc3RyaWRlID0gKDIsIDEpCiAgICAgICAgc2VsZi5sYXllcjRbMF0uZG93bnNhbXBsZVswXS5zdHJpZGUgPSAoMiwgMSkKCiAgICAgICAgc2VsZi5wcm9qID0gbm4uQ29udjJkKGZpbmFsX2NoYW5uZWxzLCBvdXRfZGltLCBrZXJuZWxfc2l6ZT0xKQoKICAgICAgICBpZiBmcmVlemVfYmFja2JvbmU6CiAgICAgICAgICAgIGZvciBuYW1lLCBwIGluIHNlbGYubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgICAgICAgICAgaWYgInByb2oiIG5vdCBpbiBuYW1lOgogICAgICAgICAgICAgICAgICAgIHAucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeCA9IHNlbGYuY29udjEoeCkKICAgICAgICB4ID0gc2VsZi5ibjEoeCkKICAgICAgICB4ID0gc2VsZi5yZWx1KHgpCiAgICAgICAgeCA9IHNlbGYubWF4cG9vbCh4KQogICAgICAgIHggPSBzZWxmLmxheWVyMSh4KQogICAgICAgIHggPSBzZWxmLmxheWVyMih4KQogICAgICAgIHggPSBzZWxmLmxheWVyMyh4KQogICAgICAgIHggPSBzZWxmLmxheWVyNCh4KQogICAgICAgIHggPSBzZWxmLnByb2ooeCkKICAgICAgICB4ID0gRi5hZGFwdGl2ZV9hdmdfcG9vbDJkKHgsICgxLCBOb25lKSkKICAgICAgICByZXR1cm4geAoKCmNsYXNzIENvbnZOZVh0RXh0cmFjdG9yKG5uLk1vZHVsZSk6CiAgICAiIiJDb252TmVYdCB0aW55L2Jhc2UgYmFja2JvbmUgd2l0aCBoZWlnaHQtY29sbGFwc2luZyBzdHJpZGUgbW9kaWZpY2F0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2YXJpYW50PSJjb252bmV4dF90aW55IiwgcHJldHJhaW5lZD1UcnVlLCBvdXRfZGltPTUxMiwgZnJlZXplX2JhY2tib25lPUZhbHNlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiB2YXJpYW50ID09ICJjb252bmV4dF90aW55IjoKICAgICAgICAgICAgYmFja2JvbmUgPSBjb252bmV4dF90aW55KHdlaWdodHM9Q29udk5lWHRfVGlueV9XZWlnaHRzLklNQUdFTkVUMUtfVjEgaWYgcHJldHJhaW5lZCBlbHNlIE5vbmUpCiAgICAgICAgICAgIGZpbmFsX2NoYW5uZWxzID0gNzY4CiAgICAgICAgZWxpZiB2YXJpYW50ID09ICJjb252bmV4dF9iYXNlIjoKICAgICAgICAgICAgYmFja2JvbmUgPSBjb252bmV4dF9iYXNlKHdlaWdodHM9Q29udk5lWHRfQmFzZV9XZWlnaHRzLklNQUdFTkVUMUtfVjEgaWYgcHJldHJhaW5lZCBlbHNlIE5vbmUpCiAgICAgICAgICAgIGZpbmFsX2NoYW5uZWxzID0gMTAyNAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmtub3duIGNvbnZuZXh0IHZhcmlhbnQ6IHt2YXJpYW50fSIpCgogICAgICAgIHNlbGYuZmVhdHVyZXMgPSBiYWNrYm9uZS5mZWF0dXJlcwoKICAgICAgICAjIERvd25zYW1wbGluZyBjb252cyBhdCBpbmRpY2VzIDQgYW5kIDYgdXNlIGtlcm5lbD0yLCBzdHJpZGU9Mi4KICAgICAgICAjIE1ha2UgdGhlbSAoMiwxKS8oMiwxKSBzbyBoZWlnaHQgaGFsdmVzIGJ1dCB3aWR0aCBpcyBwcmVzZXJ2ZWQuCiAgICAgICAgZm9yIGRzX2lkeCBpbiBbNCwgNl06CiAgICAgICAgICAgIGNvbnYgPSBzZWxmLmZlYXR1cmVzW2RzX2lkeF1bMV0KICAgICAgICAgICAgY29udi5zdHJpZGUgPSAoMiwgMSkKICAgICAgICAgICAgY29udi5rZXJuZWxfc2l6ZSA9ICgyLCAxKQogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIGNvbnYud2VpZ2h0ID0gbm4uUGFyYW1ldGVyKGNvbnYud2VpZ2h0WzosIDosIDosIDoxXS5jb250aWd1b3VzKCkpCgogICAgICAgIHNlbGYucHJvaiA9IG5uLkNvbnYyZChmaW5hbF9jaGFubmVscywgb3V0X2RpbSwga2VybmVsX3NpemU9MSkKCiAgICAgICAgaWYgZnJlZXplX2JhY2tib25lOgogICAgICAgICAgICBmb3IgbmFtZSwgcCBpbiBzZWxmLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICAgICAgICAgIGlmICJwcm9qIiBub3QgaW4gbmFtZToKICAgICAgICAgICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHggPSBzZWxmLmZlYXR1cmVzKHgpCiAgICAgICAgeCA9IHNlbGYucHJvaih4KQogICAgICAgIHggPSBGLmFkYXB0aXZlX2F2Z19wb29sMmQoeCwgKDEsIE5vbmUpKQogICAgICAgIHJldHVybiB4CgoKX1JFU05FVCA9IHsicmVzbmV0MzQiLCAicmVzbmV0NTAifQpfQ09OVk5FWFQgPSB7ImNvbnZuZXh0X3RpbnkiLCAiY29udm5leHRfYmFzZSJ9CgoKZGVmIGJ1aWxkX2JhY2tib25lKG5hbWU9InJlc25ldDUwIiwgcHJldHJhaW5lZD1UcnVlLCBvdXRfZGltPTUxMiwgZnJlZXplX2JhY2tib25lPUZhbHNlKToKICAgICIiIkZhY3RvcnkgcmV0dXJuaW5nIGEgZmVhdHVyZSBleHRyYWN0b3IgZm9yIHRoZSByZXF1ZXN0ZWQgYmFja2JvbmUgbmFtZS4iIiIKICAgIGlmIG5hbWUgaW4gX1JFU05FVDoKICAgICAgICByZXR1cm4gUmVzTmV0RXh0cmFjdG9yKG5hbWUsIHByZXRyYWluZWQsIG91dF9kaW0sIGZyZWV6ZV9iYWNrYm9uZSkKICAgIGlmIG5hbWUgaW4gX0NPTlZORVhUOgogICAgICAgIHJldHVybiBDb252TmVYdEV4dHJhY3RvcihuYW1lLCBwcmV0cmFpbmVkLCBvdXRfZGltLCBmcmVlemVfYmFja2JvbmUpCiAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgIGYiVW5rbm93biBiYWNrYm9uZSAne25hbWV9Jy4gQXZhaWxhYmxlOiB7c29ydGVkKF9SRVNORVQgfCBfQ09OVk5FWFQpfSIKICAgICkK",
"models/components.py": "aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYNCmZyb20gdG9yY2h2aXNpb24ubW9kZWxzIGltcG9ydCByZXNuZXQ1MCwgUmVzTmV0NTBfV2VpZ2h0cw0KDQoNCmNsYXNzIFNUTkJsb2NrKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubG9jYWxpemF0aW9uID0gbm4uU2VxdWVudGlhbCgNCiAgICAgICAgICAgIG5uLkNvbnYyZChpbl9jaGFubmVscywgMzIsIGtlcm5lbF9zaXplPTUsIHN0cmlkZT0yLCBwYWRkaW5nPTIpLA0KICAgICAgICAgICAgbm4uTWF4UG9vbDJkKDIsIDIpLA0KICAgICAgICAgICAgbm4uUmVMVShUcnVlKSwNCiAgICAgICAgICAgIG5uLkNvbnYyZCgzMiwgNjQsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT0xLCBwYWRkaW5nPTEpLA0KICAgICAgICAgICAgbm4uUmVMVShUcnVlKSwNCiAgICAgICAgICAgIG5uLkFkYXB0aXZlQXZnUG9vbDJkKCg0LCA4KSkNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLmZjX2xvYyA9IG5uLlNlcXVlbnRpYWwoDQogICAgICAgICAgICBubi5GbGF0dGVuKCksDQogICAgICAgICAgICBubi5MaW5lYXIoNjQgKiA0ICogOCwgMTI4KSwNCiAgICAgICAgICAgIG5uLlJlTFUoVHJ1ZSksDQogICAgICAgICAgICBubi5MaW5lYXIoMTI4LCA2KQ0KICAgICAgICApDQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgc2VsZi5mY19sb2NbLTFdLndlaWdodC56ZXJvXygpDQogICAgICAgICAgICBzZWxmLmZjX2xvY1stMV0uYmlhcy5jb3B5XygNCiAgICAgICAgICAgICAgICB0b3JjaC50ZW5zb3IoWzEsIDAsIDAsIDAsIDEsIDBdLCBkdHlwZT10b3JjaC5mbG9hdCkNCiAgICAgICAgICAgICkNCiAgICANCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgeCA9IHNlbGYubG9jYWxpemF0aW9uKHgpDQogICAgICAgIHggPSBzZWxmLmZjX2xvYyh4KQ0KICAgICAgICB4ID0geC52aWV3KC0xLCAyLCAzKQ0KICAgICAgICByZXR1cm4geA0KICAgIA0KDQpjbGFzcyBBdHRlbnRpb25GdXNpb24obm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2hhbm5lbHMpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5zY29yZV9uZXQgPSBubi5TZXF1ZW50aWFsKA0KICAgICAgICAgICAgbm4uQ29udjJkKGNoYW5uZWxzLCBjaGFubmVscyAvLyA4LCBrZXJuZWxfc2l6ZT0xKSwNCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwNCiAgICAgICAgICAgIG5uLkNvbnYyZChjaGFubmVscyAvLyA4LCAxLCBrZXJuZWxfc2l6ZT0xKQ0KICAgICAgICApDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgdG90YWxfZnJhbWVzLCBDLCBILCBXID0geC5zaXplKCkNCiAgICAgICAgbnVtX2ZyYW1lcyA9IDUNCiAgICAgICAgYmF0Y2hfc2l6ZSA9IHRvdGFsX2ZyYW1lcyAvLyBudW1fZnJhbWVzDQoNCiAgICAgICAgIyBSZXNoYXBlIHRvIFtCYXRjaF9zaXplLCBGcmFtZXMsIEMsIEgsIFddDQogICAgICAgIHhfdmlldyA9IHgudmlldyhiYXRjaF9zaXplLCBudW1fZnJhbWVzLCBDLCBILCBXKQ0KDQogICAgICAgICMgQ2FsY3VsYXRlIGF0dGVudGlvbiBzY29yZXMgW0JhdGNoX3NpemUsIEZyYW1lcywgMSwgSCwgV10NCiAgICAgICAgc2NvcmVzID0gc2VsZi5zY29yZV9uZXQoeCkudmlldyhiYXRjaF9zaXplLCBudW1fZnJhbWVzLCAxLCBILCBXKQ0KICAgICAgICB3ZWlnaHRzID0gRi5zb2Z0bWF4KHNjb3JlcywgZGltPTEpDQoNCiAgICAgICAgIyBXZWlnaHQgc2ltIGZ1c2lvbg0KICAgICAgICBmdXNlZF9mZWF0dXJlcyA9IHRvcmNoLnN1bSh4X3ZpZXcgKiB3ZWlnaHRzLCBkaW09MSkNCiAgICAgICAgcmV0dXJuIGZ1c2VkX2ZlYXR1cmVzDQoNCg0KY2xhc3MgRmVhdHVyZUV4dHJhY3Rvcihubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwcmV0cmFpbmVkPVRydWUsIG91dF9kaW09NTEyLCBmcmVlemVfYmFja2JvbmU9RmFsc2UpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgd2VpZ2h0cyA9IFJlc05ldDUwX1dlaWdodHMuSU1BR0VORVQxS19WMiBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZQ0KICAgICAgICBiYWNrYm9uZSA9IHJlc25ldDUwKHdlaWdodHM9d2VpZ2h0cykNCg0KICAgICAgICBzZWxmLmNvbnYxICAgPSBiYWNrYm9uZS5jb252MQ0KICAgICAgICBzZWxmLmJuMSAgICAgPSBiYWNrYm9uZS5ibjENCiAgICAgICAgc2VsZi5yZWx1ICAgID0gYmFja2JvbmUucmVsdQ0KICAgICAgICBzZWxmLm1heHBvb2wgPSBiYWNrYm9uZS5tYXhwb29sDQogICAgICAgIHNlbGYubGF5ZXIxICA9IGJhY2tib25lLmxheWVyMQ0KICAgICAgICBzZWxmLmxheWVyMiAgPSBiYWNrYm9uZS5sYXllcjINCiAgICAgICAgc2VsZi5sYXllcjMgID0gYmFja2JvbmUubGF5ZXIzDQogICAgICAgIHNlbGYubGF5ZXI0ICA9IGJhY2tib25lLmxheWVyNA0KDQogICAgICAgICMgTW9kaWZ5IHN0cmlkZSAoMiwyKSAtPiAoMiwxKSBpbiBsYXllcjMgdsOgIGxheWVyNCwga2VlcCB3LCBzaHJpbmsgaCBvbmx5DQoNCiAgICAgICAgc2VsZi5sYXllcjNbMF0uY29udjIuc3RyaWRlID0gKDIsIDEpDQogICAgICAgIHNlbGYubGF5ZXIzWzBdLmRvd25zYW1wbGVbMF0uc3RyaWRlID0gKDIsIDEpDQoNCiAgICAgICAgc2VsZi5sYXllcjRbMF0uY29udjIuc3RyaWRlID0gKDIsIDEpDQogICAgICAgIHNlbGYubGF5ZXI0WzBdLmRvd25zYW1wbGVbMF0uc3RyaWRlID0gKDIsIDEpDQoNCiAgICAgICAgIyBQcm9qZWN0IDIwNDggLT4gb3V0X2RpbQ0KICAgICAgICBzZWxmLnByb2ogPSBubi5Db252MmQoMjA0OCwgb3V0X2RpbSwga2VybmVsX3NpemU9MSkNCg0KICAgICAgICBpZiBmcmVlemVfYmFja2JvbmU6DQogICAgICAgICAgICBmb3IgbmFtZSwgcCBpbiBzZWxmLm5hbWVkX3BhcmFtZXRlcnMoKToNCiAgICAgICAgICAgICAgICBpZiAncHJvaicgbm90IGluIG5hbWU6DQogICAgICAgICAgICAgICAgICAgIHAucmVxdWlyZXNfZ3JhZCA9IEZhbHNlDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgeCA9IHNlbGYuY29udjEoeCkNCiAgICAgICAgeCA9IHNlbGYuYm4xKHgpDQogICAgICAgIHggPSBzZWxmLnJlbHUoeCkNCiAgICAgICAgeCA9IHNlbGYubWF4cG9vbCh4KQ0KICAgICAgICB4ID0gc2VsZi5sYXllcjEoeCkNCiAgICAgICAgeCA9IHNlbGYubGF5ZXIyKHgpDQogICAgICAgIHggPSBzZWxmLmxheWVyMyh4KQ0KICAgICAgICB4ID0gc2VsZi5sYXllcjQoeCkNCiAgICAgICAgeCA9IHNlbGYucHJvaih4KSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgKEIsIG91dF9kaW0sIEgnLCBXJykNCg0KICAgICAgICAjIENvbGxhcHNlIGhlaWdodCDihpIgMSwgZ2nhu68gd2lkdGggbMOgbSBzZXF1ZW5jZQ0KICAgICAgICB4ID0gRi5hZGFwdGl2ZV9hdmdfcG9vbDJkKHgsICgxLCBOb25lKSkgICAgICAgIyAoQiwgb3V0X2RpbSwgMSwgVycpDQogICAgICAgIHJldHVybiB4DQogICAgDQoNCg0KY2xhc3MgcG9zX2VuY29kaW5nKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkX2RpbSwgbWF4X2xlbmd0aD01MDApOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5wb3NfZW5jb2RpbmcgPSBubi5QYXJhbWV0ZXIoDQogICAgICAgICAgICAoZW1iZWRfZGltICoqIC0wLjUpICogdG9yY2gucmFuZG4oMSwgbWF4X2xlbmd0aCwgZW1iZWRfZGltKQ0KICAgICAgICApDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgQiwgVCwgQyA9IHguc2hhcGUNCiAgICAgICAgcmV0dXJuIHggKyBzZWxmLnBvc19lbmNvZGluZ1s6LCA6VCwgOl0NCiAgICANCg0KDQpjbGFzcyBUcmFuc2Zvcm1lckVuY29kZXJCbG9jayhubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlbWJlZF9kaW0sIGZmX2RpbSwgbnVtX2hlYWRzLCBkcm9wX291dCk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLmF0dG4gPSBubi5NdWx0aWhlYWRBdHRlbnRpb24oZW1iZWRfZGltLCBudW1faGVhZHMsIGRyb3BvdXQ9ZHJvcF9vdXQsIGJhdGNoX2ZpcnN0PVRydWUpDQogICAgICAgIHNlbGYuZmYgPSBubi5TZXF1ZW50aWFsKA0KICAgICAgICAgICAgbm4uTGluZWFyKGVtYmVkX2RpbSwgZmZfZGltKSwNCiAgICAgICAgICAgIG5uLkdFTFUoKSwNCiAgICAgICAgICAgIG5uLkxpbmVhcihmZl9kaW0sIGVtYmVkX2RpbSkNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLmxheWVybm9ybTEgPSBubi5MYXllck5vcm0oZW1iZWRfZGltLCBlcHM9MWUtNikNCiAgICAgICAgc2VsZi5sYXllcm5vcm0yID0gbm4uTGF5ZXJOb3JtKGVtYmVkX2RpbSwgZXBzPTFlLTYpDQogICAgICAgIHNlbGYuZHJvcDEgPSBubi5Ecm9wb3V0KGRyb3Bfb3V0KQ0KICAgICAgICBzZWxmLmRyb3AyID0gbm4uRHJvcG91dChkcm9wX291dCkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHEsIGssIHYpOg0KICAgICAgICBxX25vcm0gPSBzZWxmLmxheWVybm9ybTEocSkNCiAgICAgICAga19ub3JtID0gc2VsZi5sYXllcm5vcm0xKGspDQogICAgICAgIHZfbm9ybSA9IHNlbGYubGF5ZXJub3JtMSh2KQ0KDQogICAgICAgIGF0dG5fb3V0LCBfID0gc2VsZi5hdHRuKHFfbm9ybSwga19ub3JtLCB2X25vcm0pDQogICAgICAgIGF0dG5fb3V0ID0gc2VsZi5kcm9wMShhdHRuX291dCkNCiAgICAgICAgb3V0MSA9IGF0dG5fb3V0ICsgcQ0KDQogICAgICAgIG91dDFfbm9ybSA9IHNlbGYubGF5ZXJub3JtMihvdXQxKQ0KICAgICAgICBmZl9vdXQgPSBzZWxmLmZmKG91dDFfbm9ybSkNCiAgICAgICAgZmZfb3V0ID0gc2VsZi5kcm9wMihmZl9vdXQpDQogICAgICAgIG91dDIgPSBmZl9vdXQgKyBvdXQxDQoNCiAgICAgICAgcmV0dXJuIG91dDINCiAgICANCg0KDQpjbGFzcyBUcmFuc2Zvcm1lckVuY29kZXIobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgZW1iZWRfZGltLCBmZl9kaW0sIG51bV9sYXllcnMsIG51bV9oZWFkcywgZHJvcF9vdXQpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCg0KICAgICAgICBzZWxmLmJsb2NrcyA9IG5uLk1vZHVsZUxpc3QoWw0KICAgICAgICAgICAgVHJhbnNmb3JtZXJFbmNvZGVyQmxvY2soDQogICAgICAgICAgICAgICAgZW1iZWRfZGltLCBmZl9kaW0sIG51bV9oZWFkcywgZHJvcF9vdXQNCiAgICAgICAgICAgICkgZm9yIF8gaW4gcmFuZ2UobnVtX2xheWVycykNCiAgICAgICAgXSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOg0KICAgICAgICBvdXRwdXQgPSB4DQogICAgICAgIGZvciBibG9jayBpbiBzZWxmLmJsb2NrczoNCiAgICAgICAgICAgIG91dHB1dCA9IGJsb2NrKG91dHB1dCwgb3V0cHV0LCBvdXRwdXQpDQogICAgICAgIHJldHVybiBvdXRwdXQ=",
"models/decoders.py": "IiIiU2VxdWVuY2UtZGVjb2RlciByZWdpc3RyeSAoVHJhbnNmb3JtZXIgLyBCaUxTVE0pLgoKRWFjaCBkZWNvZGVyIG1hcHMgYSB3aWR0aC1zZXF1ZW5jZSBgYChCLCBULCBlbWJlZF9kaW0pYGAgdG8gYGAoQiwgVCwgZW1iZWRfZGltKWBgLgpUaGUgdHJhbnNmb3JtZXIgZGVjb2RlciBrZWVwcyBgYHBvc19lbmNvZGVyYGAgYW5kIGBgZW5jb2RlcmBgIGFzIHN1Ym1vZHVsZXMgc28KbGVnYWN5IG5vdGVib29rIGNoZWNrcG9pbnRzIChgYHBvc19lbmNvZGVyLipgYCAvIGBgdHJhbnNmb3JtZXJfbGF5ZXIuKmBgKSBjYW4gYmUKcmVtYXBwZWQgb250byBpdCAoc2VlIGBgdXRpbHMuY2hlY2twb2ludC5sb2FkX2NoZWNrcG9pbnRgYCkuCiIiIgoKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpmcm9tIC5jb21wb25lbnRzIGltcG9ydCBUcmFuc2Zvcm1lckVuY29kZXIsIHBvc19lbmNvZGluZwoKCmNsYXNzIFRyYW5zZm9ybWVyRGVjb2Rlcihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkX2RpbSwgZmZfZGltLCBudW1fbGF5ZXJzLCBudW1faGVhZHMsIGRyb3Bfb3V0PTAuMSwgbWF4X2xlbmd0aD01MDAwKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnBvc19lbmNvZGVyID0gcG9zX2VuY29kaW5nKGVtYmVkX2RpbSwgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoKQogICAgICAgIHNlbGYuZW5jb2RlciA9IFRyYW5zZm9ybWVyRW5jb2RlcihlbWJlZF9kaW0sIGZmX2RpbSwgbnVtX2xheWVycywgbnVtX2hlYWRzLCBkcm9wX291dCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4ID0gc2VsZi5wb3NfZW5jb2Rlcih4KQogICAgICAgIHJldHVybiBzZWxmLmVuY29kZXIoeCkKCgpjbGFzcyBCaUxTVE1EZWNvZGVyKG5uLk1vZHVsZSk6CiAgICAiIiIzLWxheWVyIGJpZGlyZWN0aW9uYWwgTFNUTTsgY29uY2F0IG9mIGJvdGggZGlyZWN0aW9ucyA9IGVtYmVkX2RpbS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZW1iZWRfZGltLCBudW1fbGF5ZXJzPTMsIGRyb3Bfb3V0PTAuMSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5sc3RtID0gbm4uTFNUTSgKICAgICAgICAgICAgaW5wdXRfc2l6ZT1lbWJlZF9kaW0sCiAgICAgICAgICAgIGhpZGRlbl9zaXplPWVtYmVkX2RpbSAvLyAyLAogICAgICAgICAgICBudW1fbGF5ZXJzPW51bV9sYXllcnMsCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0PVRydWUsCiAgICAgICAgICAgIGJpZGlyZWN0aW9uYWw9VHJ1ZSwKICAgICAgICAgICAgZHJvcG91dD1kcm9wX291dCBpZiBudW1fbGF5ZXJzID4gMSBlbHNlIDAuMCwKICAgICAgICApCiAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5ZXJOb3JtKGVtYmVkX2RpbSkKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGRyb3Bfb3V0KQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIG91dCwgXyA9IHNlbGYubHN0bSh4KQogICAgICAgIG91dCA9IHNlbGYuZHJvcG91dChvdXQpCiAgICAgICAgb3V0ID0gc2VsZi5ub3JtKG91dCkKICAgICAgICByZXR1cm4gb3V0CgoKZGVmIGJ1aWxkX2RlY29kZXIobmFtZSwgZW1iZWRfZGltLCBmZl9kaW0sIG51bV9sYXllcnMsIG51bV9oZWFkcywgZHJvcF9vdXQ9MC4xKToKICAgIG5hbWUgPSBuYW1lLmxvd2VyKCkKICAgIGlmIG5hbWUgPT0gInRyYW5zZm9ybWVyIjoKICAgICAgICByZXR1cm4gVHJhbnNmb3JtZXJEZWNvZGVyKGVtYmVkX2RpbSwgZmZfZGltLCBudW1fbGF5ZXJzLCBudW1faGVhZHMsIGRyb3Bfb3V0KQogICAgaWYgbmFtZSA9PSAiYmlsc3RtIjoKICAgICAgICByZXR1cm4gQmlMU1RNRGVjb2RlcihlbWJlZF9kaW0sIG51bV9sYXllcnMsIGRyb3Bfb3V0KQogICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gZGVjb2RlciAne25hbWV9Jy4gQXZhaWxhYmxlOiB0cmFuc2Zvcm1lciwgYmlsc3RtIikK",
"models/fusion.py": "IiIiTXVsdGktZnJhbWUgZnVzaW9uIHJlZ2lzdHJ5LgoKRWFjaCBmdXNpb24gbW9kdWxlIHRha2VzIHBlci1mcmFtZSBmZWF0dXJlIG1hcHMgb2Ygc2hhcGUgYGAoQipGLCBDLCBILCBXKWBgIGFuZApyZXR1cm5zIGEgc2luZ2xlIGZ1c2VkIG1hcCBgYChCLCBDLCBILCBXKWBgLiBgYG51bV9mcmFtZXNgYCBpcyBwYXNzZWQgYXQgY2FsbAp0aW1lIHNvIHRoZSBtb2R1bGVzIGFyZSBub3QgdGllZCB0byBhIGZpeGVkIGZyYW1lIGNvdW50LgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmNsYXNzIE1lYW5GdXNpb24obm4uTW9kdWxlKToKICAgICIiIlNpbXBsZXN0IGJhc2VsaW5lOiBhdmVyYWdlIHRoZSBmcmFtZXMuIiIiCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCwgbnVtX2ZyYW1lcyk6CiAgICAgICAgYmYsIEMsIEgsIFcgPSB4LnNpemUoKQogICAgICAgIHggPSB4LnZpZXcoYmYgLy8gbnVtX2ZyYW1lcywgbnVtX2ZyYW1lcywgQywgSCwgVykKICAgICAgICByZXR1cm4geC5tZWFuKGRpbT0xKQoKCmNsYXNzIE1heEZ1c2lvbihubi5Nb2R1bGUpOgogICAgIiIiRWxlbWVudC13aXNlIG1heCBvdmVyIGZyYW1lcy4iIiIKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBudW1fZnJhbWVzKToKICAgICAgICBiZiwgQywgSCwgVyA9IHguc2l6ZSgpCiAgICAgICAgeCA9IHgudmlldyhiZiAvLyBudW1fZnJhbWVzLCBudW1fZnJhbWVzLCBDLCBILCBXKQogICAgICAgIHJldHVybiB4Lm1heChkaW09MSkudmFsdWVzCgoKY2xhc3MgQXR0ZW50aW9uRnVzaW9uKG5uLk1vZHVsZSk6CiAgICAiIiJPcmlnaW5hbCBiYXNlbGluZTogcGVyLXBpeGVsIHNvZnRtYXggd2VpZ2h0aW5nIGFjcm9zcyBmcmFtZXMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnNjb3JlX25ldCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZChjaGFubmVscywgY2hhbm5lbHMgLy8gOCwga2VybmVsX3NpemU9MSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKGNoYW5uZWxzIC8vIDgsIDEsIGtlcm5lbF9zaXplPTEpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBudW1fZnJhbWVzKToKICAgICAgICBiZiwgQywgSCwgVyA9IHguc2l6ZSgpCiAgICAgICAgYmF0Y2hfc2l6ZSA9IGJmIC8vIG51bV9mcmFtZXMKICAgICAgICB4X3ZpZXcgPSB4LnZpZXcoYmF0Y2hfc2l6ZSwgbnVtX2ZyYW1lcywgQywgSCwgVykKICAgICAgICBzY29yZXMgPSBzZWxmLnNjb3JlX25ldCh4KS52aWV3KGJhdGNoX3NpemUsIG51bV9mcmFtZXMsIDEsIEgsIFcpCiAgICAgICAgd2VpZ2h0cyA9IEYuc29mdG1heChzY29yZXMsIGRpbT0xKQogICAgICAgIHJldHVybiB0b3JjaC5zdW0oeF92aWV3ICogd2VpZ2h0cywgZGltPTEpCgoKY2xhc3MgRnJhbWVRdWFsaXR5RnVzaW9uKG5uLk1vZHVsZSk6CiAgICAiIiJPbmUgc2NhbGFyIHdlaWdodCBwZXIgZnJhbWUgKGdsb2JhbCksIGkuZS4gc29mdCBiZXN0LWZyYW1lIHNlbGVjdGlvbi4KCiAgICBBIGZyYW1lJ3Mgd2hvbGUgZmVhdHVyZSBtYXAgaXMgZ2xvYmFsbHkgcG9vbGVkIHRvIGEgcXVhbGl0eSBzY29yZSwgdGhlbiBhCiAgICBzb2Z0bWF4IG92ZXIgZnJhbWVzIHByb2R1Y2VzIHBlci1mcmFtZSB3ZWlnaHRzIHNoYXJlZCBhY3Jvc3MgYWxsIHBpeGVscy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaGFubmVscyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5zY29yZV9uZXQgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgxKSwKICAgICAgICAgICAgbm4uRmxhdHRlbigpLAogICAgICAgICAgICBubi5MaW5lYXIoY2hhbm5lbHMsIGNoYW5uZWxzIC8vIDgpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkxpbmVhcihjaGFubmVscyAvLyA4LCAxKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCwgbnVtX2ZyYW1lcyk6CiAgICAgICAgYmYsIEMsIEgsIFcgPSB4LnNpemUoKQogICAgICAgIGJhdGNoX3NpemUgPSBiZiAvLyBudW1fZnJhbWVzCiAgICAgICAgeF92aWV3ID0geC52aWV3KGJhdGNoX3NpemUsIG51bV9mcmFtZXMsIEMsIEgsIFcpCiAgICAgICAgc2NvcmVzID0gc2VsZi5zY29yZV9uZXQoeCkudmlldyhiYXRjaF9zaXplLCBudW1fZnJhbWVzLCAxLCAxLCAxKQogICAgICAgIHdlaWdodHMgPSBGLnNvZnRtYXgoc2NvcmVzLCBkaW09MSkKICAgICAgICByZXR1cm4gdG9yY2guc3VtKHhfdmlldyAqIHdlaWdodHMsIGRpbT0xKQoKCmNsYXNzIFRlbXBvcmFsVHJhbnNmb3JtZXJGdXNpb24obm4uTW9kdWxlKToKICAgICIiIlNlbGYtYXR0ZW50aW9uIGFjcm9zcyB0aGUgNSBmcmFtZXMgYXQgZWFjaCBzcGF0aWFsIGxvY2F0aW9uLgoKICAgIFVubGlrZSBBdHRlbnRpb25GdXNpb24gKGluZGVwZW5kZW50IHBlci1mcmFtZSBzY29yZXMpLCB0aGlzIGxldHMgZnJhbWVzCiAgICBhdHRlbmQgdG8gZWFjaCBvdGhlciBiZWZvcmUgYmVpbmcgcG9vbGVkLCBtb2RlbGxpbmcgaW50ZXItZnJhbWUgcmVsYXRpb25zLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzLCBudW1faGVhZHM9NCwgZHJvcF9vdXQ9MC4xKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmF0dG4gPSBubi5NdWx0aWhlYWRBdHRlbnRpb24oY2hhbm5lbHMsIG51bV9oZWFkcywgZHJvcG91dD1kcm9wX291dCwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oY2hhbm5lbHMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCwgbnVtX2ZyYW1lcyk6CiAgICAgICAgYmYsIEMsIEgsIFcgPSB4LnNpemUoKQogICAgICAgIGJhdGNoX3NpemUgPSBiZiAvLyBudW1fZnJhbWVzCiAgICAgICAgIyAoQiwgRiwgQywgSCwgVykgLT4gKEIqSCpXLCBGLCBDKTogZWFjaCBzcGF0aWFsIGxvY2F0aW9uIGlzIGEgc2VxdWVuY2Ugb2YgZnJhbWVzCiAgICAgICAgeF92aWV3ID0geC52aWV3KGJhdGNoX3NpemUsIG51bV9mcmFtZXMsIEMsIEgsIFcpCiAgICAgICAgc2VxID0geF92aWV3LnBlcm11dGUoMCwgMywgNCwgMSwgMikucmVzaGFwZShiYXRjaF9zaXplICogSCAqIFcsIG51bV9mcmFtZXMsIEMpCiAgICAgICAgc2VxX25vcm0gPSBzZWxmLm5vcm0oc2VxKQogICAgICAgIGF0dG5fb3V0LCBfID0gc2VsZi5hdHRuKHNlcV9ub3JtLCBzZXFfbm9ybSwgc2VxX25vcm0pCiAgICAgICAgc2VxID0gc2VxICsgYXR0bl9vdXQKICAgICAgICBmdXNlZCA9IHNlcS5tZWFuKGRpbT0xKSAgIyBwb29sIG92ZXIgZnJhbWVzCiAgICAgICAgZnVzZWQgPSBmdXNlZC52aWV3KGJhdGNoX3NpemUsIEgsIFcsIEMpLnBlcm11dGUoMCwgMywgMSwgMikuY29udGlndW91cygpCiAgICAgICAgcmV0dXJuIGZ1c2VkCgoKZGVmIGJ1aWxkX2Z1c2lvbihuYW1lPSJhdHRlbnRpb24iLCBjaGFubmVscz01MTIsIG51bV9oZWFkcz00LCBkcm9wX291dD0wLjEpOgogICAgbmFtZSA9IG5hbWUubG93ZXIoKQogICAgaWYgbmFtZSA9PSAibWVhbiI6CiAgICAgICAgcmV0dXJuIE1lYW5GdXNpb24oKQogICAgaWYgbmFtZSA9PSAibWF4IjoKICAgICAgICByZXR1cm4gTWF4RnVzaW9uKCkKICAgIGlmIG5hbWUgPT0gImF0dGVudGlvbiI6CiAgICAgICAgcmV0dXJuIEF0dGVudGlvbkZ1c2lvbihjaGFubmVscykKICAgIGlmIG5hbWUgPT0gImZyYW1lX3F1YWxpdHkiOgogICAgICAgIHJldHVybiBGcmFtZVF1YWxpdHlGdXNpb24oY2hhbm5lbHMpCiAgICBpZiBuYW1lID09ICJ0ZW1wb3JhbF90cmFuc2Zvcm1lciI6CiAgICAgICAgcmV0dXJuIFRlbXBvcmFsVHJhbnNmb3JtZXJGdXNpb24oY2hhbm5lbHMsIG51bV9oZWFkcywgZHJvcF9vdXQpCiAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgIGYiVW5rbm93biBmdXNpb24gJ3tuYW1lfScuIEF2YWlsYWJsZTogbWVhbiwgbWF4LCBhdHRlbnRpb24sICIKICAgICAgICBmImZyYW1lX3F1YWxpdHksIHRlbXBvcmFsX3RyYW5zZm9ybWVyIgogICAgKQo=",
"models/restransORC.py": "IiIiQ29uZmlndXJhYmxlIG11bHRpLWZyYW1lIE9DUiBtb2RlbCAocmVnaXN0cnktYmFzZWQpLg0KDQpDb21wb3NpdGlvbjogIFNUTiAtPiBbU1JdIC0+IGJhY2tib25lIC0+IGZ1c2lvbiAtPiBkZWNvZGVyIC0+IGhlYWQgKCsgb3B0aW9uYWwNCmxheW91dCBoZWFkKS4gRXZlcnkgY29tcG9uZW50IGlzIGNob3NlbiBmcm9tIGEgcmVnaXN0cnkgdmlhIDpjbGFzczpgQ29uZmlnYCwgc28NCmFsbCA3IG9yaWdpbmFsIG5vdGVib29rcyBjb2xsYXBzZSBpbnRvIG9uZSBtb2RlbCBkZWZpbml0aW9uLg0KDQpgYGZvcndhcmQoeClgYCByZXR1cm5zIHRoZSBPQ1IgbG9naXRzIGBgKEIsIGxhYmVsX2xlbiwgbnVtX2NsYXNzZXMpYGAgc28gdGhlDQpsZWdhY3kgdHJhaW5lci9wcmVkaWN0b3Iga2VlcCB3b3JraW5nLiBgYGZvcndhcmQoeCwgcmV0dXJuX2F1eD1UcnVlKWBgIHJldHVybnMgYQ0KZGljdCBgYHtsb2dpdHMsIHNyLCBsYXlvdXR9YGAgdXNlZCBieSB0aGUgdW5pZmllZCB0cmFpbi9ldmFsIGhhcm5lc3MuDQoiIiINCg0KaW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYNCg0KZnJvbSAuYmFja2JvbmVzIGltcG9ydCBidWlsZF9iYWNrYm9uZQ0KZnJvbSAuY29tcG9uZW50cyBpbXBvcnQgU1ROQmxvY2sNCmZyb20gLmRlY29kZXJzIGltcG9ydCBidWlsZF9kZWNvZGVyDQpmcm9tIC5mdXNpb24gaW1wb3J0IGJ1aWxkX2Z1c2lvbg0KZnJvbSAuc3IgaW1wb3J0IFNSTW9kdWxlDQoNCg0KY2xhc3MgTGF5b3V0SGVhZChubi5Nb2R1bGUpOg0KICAgICIiIkNsYXNzaWZpZXMgcGxhdGUgbGF5b3V0IChlLmcuIEJyYXppbGlhbiB2cyBNZXJjb3N1cikgZnJvbSBmdXNlZCBmZWF0dXJlcy4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlbWJlZF9kaW0sIG51bV9sYXlvdXRzKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbCgNCiAgICAgICAgICAgIG5uLkFkYXB0aXZlQXZnUG9vbDJkKDEpLA0KICAgICAgICAgICAgbm4uRmxhdHRlbigpLA0KICAgICAgICAgICAgbm4uTGluZWFyKGVtYmVkX2RpbSwgZW1iZWRfZGltIC8vIDQpLA0KICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLA0KICAgICAgICAgICAgbm4uTGluZWFyKGVtYmVkX2RpbSAvLyA0LCBudW1fbGF5b3V0cyksDQogICAgICAgICkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIGZ1c2VkKToNCiAgICAgICAgcmV0dXJuIHNlbGYubmV0KGZ1c2VkKQ0KDQoNCmNsYXNzIFJlc1RyYW5PQ1Iobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbGFiZWxfbGVuLCBudW1fY2xhc3NlcywgZW1iZWRfZGltLCBmZl9kaW0sIG51bV9sYXllcnMsIG51bV9oZWFkcywNCiAgICAgICAgICAgICAgICAgYmFja2JvbmU9InJlc25ldDUwIiwgZGVjb2Rlcj0idHJhbnNmb3JtZXIiLCBmdXNpb25fdHlwZT0iYXR0ZW50aW9uIiwNCiAgICAgICAgICAgICAgICAgbnVtX2ZyYW1lcz01LCBtdWx0aV9mcmFtZT1UcnVlLA0KICAgICAgICAgICAgICAgICB1c2Vfc3I9RmFsc2UsIHNyX3NjYWxlPTIsIHVzZV9sYXlvdXRfaGVhZD1GYWxzZSwgbnVtX2xheW91dHM9MiwNCiAgICAgICAgICAgICAgICAgZXh0cmFjdG9yX3ByZXRyYWluZWQ9VHJ1ZSwgZnJlZXplX2V4dHJhY3Rvcj1UcnVlLCBkcm9wX291dD0wLjEpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5sYWJlbF9sZW4gPSBsYWJlbF9sZW4NCiAgICAgICAgc2VsZi5udW1fZnJhbWVzID0gbnVtX2ZyYW1lcw0KICAgICAgICBzZWxmLm11bHRpX2ZyYW1lID0gbXVsdGlfZnJhbWUNCiAgICAgICAgc2VsZi51c2Vfc3IgPSB1c2Vfc3INCiAgICAgICAgc2VsZi5zcl9zY2FsZSA9IHNyX3NjYWxlDQogICAgICAgIHNlbGYudXNlX2xheW91dF9oZWFkID0gdXNlX2xheW91dF9oZWFkDQoNCiAgICAgICAgaWYgdXNlX3NyOg0KICAgICAgICAgICAgc2VsZi5zcl9tb2R1bGUgPSBTUk1vZHVsZShpbl9jaGFubmVscz0zLCBzY2FsZT1zcl9zY2FsZSkNCg0KICAgICAgICBzZWxmLnN0biA9IFNUTkJsb2NrKDMpDQogICAgICAgIHNlbGYuZXh0cmFjdG9yID0gYnVpbGRfYmFja2JvbmUoDQogICAgICAgICAgICBiYWNrYm9uZSwgcHJldHJhaW5lZD1leHRyYWN0b3JfcHJldHJhaW5lZCwgb3V0X2RpbT1lbWJlZF9kaW0sDQogICAgICAgICAgICBmcmVlemVfYmFja2JvbmU9ZnJlZXplX2V4dHJhY3RvciwNCiAgICAgICAgKQ0KICAgICAgICBpZiBtdWx0aV9mcmFtZToNCiAgICAgICAgICAgIHNlbGYuZnVzaW9uID0gYnVpbGRfZnVzaW9uKGZ1c2lvbl90eXBlLCBjaGFubmVscz1lbWJlZF9kaW0sIG51bV9oZWFkcz1udW1faGVhZHMsIGRyb3Bfb3V0PWRyb3Bfb3V0KQ0KICAgICAgICBzZWxmLmRlY29kZXIgPSBidWlsZF9kZWNvZGVyKGRlY29kZXIsIGVtYmVkX2RpbSwgZmZfZGltLCBudW1fbGF5ZXJzLCBudW1faGVhZHMsIGRyb3Bfb3V0KQ0KICAgICAgICBzZWxmLmhlYWQgPSBubi5MaW5lYXIoZW1iZWRfZGltLCBudW1fY2xhc3NlcykNCiAgICAgICAgaWYgdXNlX2xheW91dF9oZWFkOg0KICAgICAgICAgICAgc2VsZi5sYXlvdXRfaGVhZCA9IExheW91dEhlYWQoZW1iZWRfZGltLCBudW1fbGF5b3V0cykNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIHJldHVybl9hdXg9RmFsc2UpOg0KICAgICAgICBCLCBGcmFtZXMsIEMsIEgsIFcgPSB4LnNpemUoKQ0KICAgICAgICBpZiBub3Qgc2VsZi5tdWx0aV9mcmFtZToNCiAgICAgICAgICAgIG1pZCA9IEZyYW1lcyAvLyAyDQogICAgICAgICAgICB4ID0geFs6LCBtaWQ6bWlkICsgMV0NCiAgICAgICAgICAgIEZyYW1lcyA9IDENCiAgICAgICAgeF9mbGF0ID0geC5yZXNoYXBlKEIgKiBGcmFtZXMsIEMsIEgsIFcpDQoNCiAgICAgICAgc3Jfb3V0ID0gTm9uZQ0KICAgICAgICBpZiBzZWxmLnVzZV9zcjoNCiAgICAgICAgICAgIHNyX291dCA9IHNlbGYuc3JfbW9kdWxlKHhfZmxhdCkgICMgKEIqRiwgQywgSCpzLCBXKnMpDQogICAgICAgICAgICBvY3JfaW4gPSBGLmludGVycG9sYXRlKHNyX291dCwgc2l6ZT0oSCwgVyksIG1vZGU9ImJpbGluZWFyIiwgYWxpZ25fY29ybmVycz1GYWxzZSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIG9jcl9pbiA9IHhfZmxhdA0KDQogICAgICAgIHRoZXRhID0gc2VsZi5zdG4ob2NyX2luKQ0KICAgICAgICBncmlkID0gRi5hZmZpbmVfZ3JpZCh0aGV0YSwgb2NyX2luLnNpemUoKSwgYWxpZ25fY29ybmVycz1GYWxzZSkNCiAgICAgICAgeF9hbGlnbmVkID0gRi5ncmlkX3NhbXBsZShvY3JfaW4sIGdyaWQsIGFsaWduX2Nvcm5lcnM9RmFsc2UpDQoNCiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmV4dHJhY3Rvcih4X2FsaWduZWQpICAjIChCKkYsIGVtYmVkX2RpbSwgMSwgVycpDQoNCiAgICAgICAgaWYgc2VsZi5tdWx0aV9mcmFtZToNCiAgICAgICAgICAgIGZ1c2VkID0gc2VsZi5mdXNpb24oZmVhdHVyZXMsIEZyYW1lcykgICMgKEIsIGVtYmVkX2RpbSwgMSwgVycpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBmdXNlZCA9IGZlYXR1cmVzICAjIChCLCBlbWJlZF9kaW0sIDEsIFcnKQ0KDQogICAgICAgIHNlcV9pbnB1dCA9IGZ1c2VkLnNxdWVlemUoMikucGVybXV0ZSgwLCAyLCAxKSAgIyAoQiwgVycsIGVtYmVkX2RpbSkNCiAgICAgICAgc2VxX291dCA9IHNlbGYuZGVjb2RlcihzZXFfaW5wdXQpDQogICAgICAgIHNlcV9vdXQgPSBzZXFfb3V0LnBlcm11dGUoMCwgMiwgMSkNCiAgICAgICAgc2VxX291dCA9IEYuYWRhcHRpdmVfYXZnX3Bvb2wxZChzZXFfb3V0LCBzZWxmLmxhYmVsX2xlbikNCiAgICAgICAgc2VxX291dCA9IHNlcV9vdXQucGVybXV0ZSgwLCAyLCAxKQ0KICAgICAgICBsb2dpdHMgPSBzZWxmLmhlYWQoc2VxX291dCkgICMgKEIsIGxhYmVsX2xlbiwgbnVtX2NsYXNzZXMpDQoNCiAgICAgICAgaWYgbm90IHJldHVybl9hdXg6DQogICAgICAgICAgICByZXR1cm4gbG9naXRzDQoNCiAgICAgICAgbGF5b3V0X2xvZ2l0cyA9IHNlbGYubGF5b3V0X2hlYWQoZnVzZWQpIGlmIHNlbGYudXNlX2xheW91dF9oZWFkIGVsc2UgTm9uZQ0KICAgICAgICByZXR1cm4geyJsb2dpdHMiOiBsb2dpdHMsICJzciI6IHNyX291dCwgImxheW91dCI6IGxheW91dF9sb2dpdHN9DQoNCg0KZGVmIGJ1aWxkX21vZGVsKGNmZyk6DQogICAgIiIiQ29uc3RydWN0IGEgOmNsYXNzOmBSZXNUcmFuT0NSYCBmcm9tIGEgOmNsYXNzOmBjb25maWcuY29uZmlnLkNvbmZpZ2AuIiIiDQogICAgcmV0dXJuIFJlc1RyYW5PQ1IoDQogICAgICAgIGxhYmVsX2xlbj1jZmcubGFiZWxfbGVuLA0KICAgICAgICBudW1fY2xhc3Nlcz1jZmcubnVtX2NsYXNzZXMsDQogICAgICAgIGVtYmVkX2RpbT1jZmcuZW1iZWRfZGltLA0KICAgICAgICBmZl9kaW09Y2ZnLmZmX2RpbSwNCiAgICAgICAgbnVtX2xheWVycz1jZmcubnVtX2xheWVycywNCiAgICAgICAgbnVtX2hlYWRzPWNmZy5udW1faGVhZHMsDQogICAgICAgIGJhY2tib25lPWNmZy5iYWNrYm9uZSwNCiAgICAgICAgZGVjb2Rlcj1jZmcuZGVjb2RlciwNCiAgICAgICAgZnVzaW9uX3R5cGU9Y2ZnLmZ1c2lvbl90eXBlLA0KICAgICAgICBudW1fZnJhbWVzPWNmZy5udW1fZnJhbWVzLA0KICAgICAgICBtdWx0aV9mcmFtZT1jZmcubXVsdGlfZnJhbWUsDQogICAgICAgIHVzZV9zcj1jZmcudXNlX3NyLA0KICAgICAgICBzcl9zY2FsZT1jZmcuc3Jfc2NhbGUsDQogICAgICAgIHVzZV9sYXlvdXRfaGVhZD1jZmcudXNlX2xheW91dF9oZWFkLA0KICAgICAgICBudW1fbGF5b3V0cz1sZW4oY2ZnLmxheW91dHMpLA0KICAgICAgICBleHRyYWN0b3JfcHJldHJhaW5lZD1jZmcuZXh0cmFjdG9yX3ByZXRyYWluZWQsDQogICAgICAgIGZyZWV6ZV9leHRyYWN0b3I9Y2ZnLmZyZWV6ZV9leHRyYWN0b3IsDQogICAgICAgIGRyb3Bfb3V0PWNmZy5kcm9wX291dCwNCiAgICApDQo=",
"models/sr.py": "IiIiTGlnaHR3ZWlnaHQgc3VwZXItcmVzb2x1dGlvbiAvIHJlc3RvcmF0aW9uIG1vZHVsZSBmb3IgdGhlIG11bHRpLXRhc2sgYnJhbmNoLgoKQSBzaGFsbG93IHJlc2lkdWFsIENOTiB3aXRoIGEgUGl4ZWxTaHVmZmxlIHVwc2FtcGxlci4gSXQgZW5oYW5jZXMgZWFjaCBmcmFtZQooTFIgLT4gU1IpIGJlZm9yZSB0aGUgT0NSIGJhY2tib25lIGNvbnN1bWVzIGl0LiBEdXJpbmcgdHJhaW5pbmcgdGhlIFNSIG91dHB1dCBpcwpzdXBlcnZpc2VkIGJ5IHRoZSByZWFsIEhSIGZyYW1lcyAoTDEgbG9zcyk7IGF0IGluZmVyZW5jZSAoYmxpbmQgdGVzdCwgbm8gSFIpIGl0CnNpbXBseSBydW5zIGZvcndhcmQgdG8gc2hhcnBlbiB0aGUgaW5wdXQgdGhlIE9DUiBzZWVzLgoiIiIKCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKY2xhc3MgX1Jlc0Jsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2hhbm5lbHMpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYm9keSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZChjaGFubmVscywgY2hhbm5lbHMsIDMsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKGNoYW5uZWxzLCBjaGFubmVscywgMywgcGFkZGluZz0xKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHggKyBzZWxmLmJvZHkoeCkKCgpjbGFzcyBTUk1vZHVsZShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzPTMsIG51bV9mZWF0cz02NCwgbnVtX2Jsb2Nrcz00LCBzY2FsZT0yKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnNjYWxlID0gc2NhbGUKICAgICAgICBzZWxmLmhlYWQgPSBubi5Db252MmQoaW5fY2hhbm5lbHMsIG51bV9mZWF0cywgMywgcGFkZGluZz0xKQogICAgICAgIHNlbGYuYm9keSA9IG5uLlNlcXVlbnRpYWwoKltfUmVzQmxvY2sobnVtX2ZlYXRzKSBmb3IgXyBpbiByYW5nZShudW1fYmxvY2tzKV0pCiAgICAgICAgaWYgc2NhbGUgPiAxOgogICAgICAgICAgICBzZWxmLnVwc2FtcGxlID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgIG5uLkNvbnYyZChudW1fZmVhdHMsIG51bV9mZWF0cyAqIHNjYWxlICogc2NhbGUsIDMsIHBhZGRpbmc9MSksCiAgICAgICAgICAgICAgICBubi5QaXhlbFNodWZmbGUoc2NhbGUpLAogICAgICAgICAgICApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi51cHNhbXBsZSA9IG5uLklkZW50aXR5KCkKICAgICAgICBzZWxmLnRhaWwgPSBubi5Db252MmQobnVtX2ZlYXRzLCBpbl9jaGFubmVscywgMywgcGFkZGluZz0xKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGZlYXQgPSBzZWxmLmhlYWQoeCkKICAgICAgICBmZWF0ID0gZmVhdCArIHNlbGYuYm9keShmZWF0KQogICAgICAgIGZlYXQgPSBzZWxmLnVwc2FtcGxlKGZlYXQpCiAgICAgICAgb3V0ID0gc2VsZi50YWlsKGZlYXQpCiAgICAgICAgIyBnbG9iYWwgcmVzaWR1YWw6IGFkZCBiaWxpbmVhcmx5LXVwc2NhbGVkIGlucHV0IHNvIHRoZSBtb2R1bGUgbGVhcm5zIHRoZSBkZXRhaWwKICAgICAgICBiYXNlID0gRi5pbnRlcnBvbGF0ZSh4LCBzY2FsZV9mYWN0b3I9c2VsZi5zY2FsZSwgbW9kZT0iYmlsaW5lYXIiLCBhbGlnbl9jb3JuZXJzPUZhbHNlKSBcCiAgICAgICAgICAgIGlmIHNlbGYuc2NhbGUgPiAxIGVsc2UgeAogICAgICAgIHJldHVybiBvdXQgKyBiYXNlCg==",
"datasets/__init__.py": "ZnJvbSAuZGF0YXNldCBpbXBvcnQgSUNQUkRhdGFTZXQKZnJvbSAudHJhbnNmb3JtcyBpbXBvcnQgYnVpbGRfdHJhbnNmb3JtcywgYnVpbGRfc3JfdGFyZ2V0X3RyYW5zZm9ybQpmcm9tIC5jb2xsYXRlIGltcG9ydCAoCiAgICBjb2xsYXRlX2ZuX3RyYWluLAogICAgY29sbGF0ZV9mbl90ZXN0LAogICAgY29sbGF0ZV9mbl9ibGluZF90ZXN0LAogICAgY29sbGF0ZV9oYXJuZXNzLAopCgpfX2FsbF9fID0gWwogICAgIklDUFJEYXRhU2V0IiwKICAgICJidWlsZF90cmFuc2Zvcm1zIiwKICAgICJidWlsZF9zcl90YXJnZXRfdHJhbnNmb3JtIiwKICAgICJjb2xsYXRlX2ZuX3RyYWluIiwKICAgICJjb2xsYXRlX2ZuX3Rlc3QiLAogICAgImNvbGxhdGVfZm5fYmxpbmRfdGVzdCIsCiAgICAiY29sbGF0ZV9oYXJuZXNzIiwKXQo=",
"datasets/collate.py": "IiIiQ29sbGF0ZSBmdW5jdGlvbnMuCgpUaGUgbGVnYWN5IGBgY29sbGF0ZV9mbl90cmFpbi90ZXN0L2JsaW5kX3Rlc3RgYCBrZWVwIHRoZSBvcmlnaW5hbCBub3RlYm9vawpzaWduYXR1cmVzLiBgYGNvbGxhdGVfaGFybmVzc2BgIHJldHVybnMgYSBkaWN0IGFuZCBpcyB1c2VkIGJ5IHRoZSB1bmlmaWVkCnRyYWluL2V2YWwgaGFybmVzcyAoY2Fycnlpbmcgb3B0aW9uYWwgYGBzcl90YXJnZXRgYCBhbmQgYGBsYXlvdXRgYCkuCiIiIgoKaW1wb3J0IHRvcmNoCgpmcm9tIHV0aWxzIGltcG9ydCBlbmNvZGVfbGFiZWwKClZPQ0FCID0gIjAxMjM0NTY3ODlBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWiIKCgpkZWYgY29sbGF0ZV9mbl90cmFpbihiYXRjaCk6CiAgICBpbWFnZXMgPSB0b3JjaC5zdGFjayhbdG9yY2guc3RhY2soaXRlbVsnaW1hZ2VzJ10pIGZvciBpdGVtIGluIGJhdGNoXSkKICAgIHRhcmdldHMgPSB0b3JjaC5zdGFjayhbZW5jb2RlX2xhYmVsKGl0ZW1bJ2xhYmVsJ10sIFZPQ0FCKSBmb3IgaXRlbSBpbiBiYXRjaF0pCiAgICByZXR1cm4gaW1hZ2VzLCB0YXJnZXRzCgoKZGVmIGNvbGxhdGVfZm5fdGVzdChiYXRjaCk6CiAgICBpbWFnZXMgPSB0b3JjaC5zdGFjayhbdG9yY2guc3RhY2soaXRlbVsnaW1hZ2VzJ10pIGZvciBpdGVtIGluIGJhdGNoXSkKICAgIHRhcmdldHMgPSB0b3JjaC5zdGFjayhbZW5jb2RlX2xhYmVsKGl0ZW1bJ2xhYmVsJ10sIFZPQ0FCKSBmb3IgaXRlbSBpbiBiYXRjaF0pCiAgICB0cmFja19pZHMgPSBbaXRlbVsndHJhY2tfaWQnXSBmb3IgaXRlbSBpbiBiYXRjaF0KICAgIHJldHVybiBpbWFnZXMsIHRhcmdldHMsIHRyYWNrX2lkcwoKCmRlZiBjb2xsYXRlX2ZuX2JsaW5kX3Rlc3QoYmF0Y2gpOgogICAgaW1hZ2VzID0gdG9yY2guc3RhY2soW3RvcmNoLnN0YWNrKGl0ZW1bJ2ltYWdlcyddKSBmb3IgaXRlbSBpbiBiYXRjaF0pCiAgICB0cmFja19pZHMgPSBbaXRlbVsndHJhY2tfaWQnXSBmb3IgaXRlbSBpbiBiYXRjaF0KICAgIHJldHVybiBpbWFnZXMsIHRyYWNrX2lkcwoKCmRlZiBjb2xsYXRlX2hhcm5lc3MoYmF0Y2gsIHZvY2FiPVZPQ0FCKToKICAgIG91dCA9IHsKICAgICAgICAnaW1hZ2VzJzogdG9yY2guc3RhY2soW3RvcmNoLnN0YWNrKGl0ZW1bJ2ltYWdlcyddKSBmb3IgaXRlbSBpbiBiYXRjaF0pLAogICAgICAgICd0cmFja19pZHMnOiBbaXRlbVsndHJhY2tfaWQnXSBmb3IgaXRlbSBpbiBiYXRjaF0sCiAgICAgICAgJ2xheW91dCc6IHRvcmNoLnRlbnNvcihbaXRlbVsnbGF5b3V0J10gZm9yIGl0ZW0gaW4gYmF0Y2hdLCBkdHlwZT10b3JjaC5sb25nKSwKICAgIH0KICAgIGlmIGFsbChpdGVtLmdldCgnbGFiZWwnKSBpcyBub3QgTm9uZSBmb3IgaXRlbSBpbiBiYXRjaCk6CiAgICAgICAgb3V0Wyd0YXJnZXRzJ10gPSB0b3JjaC5zdGFjayhbZW5jb2RlX2xhYmVsKGl0ZW1bJ2xhYmVsJ10sIHZvY2FiKSBmb3IgaXRlbSBpbiBiYXRjaF0pCiAgICBpZiBhbGwoJ3NyX3RhcmdldCcgaW4gaXRlbSBmb3IgaXRlbSBpbiBiYXRjaCk6CiAgICAgICAgb3V0Wydzcl90YXJnZXQnXSA9IHRvcmNoLnN0YWNrKFt0b3JjaC5zdGFjayhpdGVtWydzcl90YXJnZXQnXSkgZm9yIGl0ZW0gaW4gYmF0Y2hdKQogICAgcmV0dXJuIG91dAo=",
"datasets/dataset.py": "IiIiVW5pZmllZCBJQ1BSIG11bHRpLWZyYW1lIGRhdGFzZXQgKGV4dHJhY3RlZCBmcm9tIHRoZSBiYXNlbGluZSBub3RlYm9va3MpLgoKRml4ZXMgQjM6IGBgdHJhY2tfaWRgYCByZXR1cm5lZCB0byB0aGUgcHJlZGljdG9yIGlzIHRoZSAqb3JpZ2luYWwqIHRyYWNrIGlkCihlLmcuIGBgdHJhY2tfMTAwMDVgYCksIG5vdCBgYHRyYWNrXzEwMDA1X2xyYGAsIHNvIHRoZSBzdWJtaXNzaW9uIENTViBpcyB2YWxpZC4KVGhlIGludGVybmFsIGBgZW50cnlfaWRgYCBzdGlsbCBjYXJyaWVzIHRoZSBgYF9scmBgL2BgX2hyYGAgc3VmZml4IG9ubHkgdG8ga2VlcAp0aGUgdHdvIHRyYWluLXRpbWUgZW50cmllcyAoTFIgb3JpZ2luYWwgKyBIUi1kZWdyYWRlZCkgZGlzdGluY3QuCgpPcHRpb25hbCBleHRyYXMgZm9yIHRoZSBuZXcgaGFybmVzczoKLSBgYHJldHVybl9zcmBgICAgLT4gYWxzbyB5aWVsZHMgYGBzcl90YXJnZXRgYCAoY2xlYW4gSFIgZnJhbWVzIGF0IHNjYWxlIHggc2l6ZSkKLSBgYHJldHVybl9sYXlvdXRgYCAtPiBhbHNvIHlpZWxkcyBgYGxheW91dGBgIGluZGV4IChCcmF6aWxpYW4vTWVyY29zdXIpCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCgppbXBvcnQgYWxidW1lbnRhdGlvbnMgYXMgQQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gUElMIGltcG9ydCBJbWFnZQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0CmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldAoKZnJvbSB1dGlscyBpbXBvcnQgZW5jb2RlX2xhYmVsCgoKY2xhc3MgSUNQUkRhdGFTZXQoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aCwgc3BsaXQ9J3RyYWluJywgdmFsX3NpemU9MC4yLCByYW5kb21fc3RhdGU9NDIsIHRyYW5zZm9ybT1Ob25lLAogICAgICAgICAgICAgICAgIHNyX3RyYW5zZm9ybT1Ob25lLCByZXR1cm5fc3I9RmFsc2UsIHJldHVybl9sYXlvdXQ9RmFsc2UsCiAgICAgICAgICAgICAgICAgbGF5b3V0cz0oJ0JyYXppbGlhbicsICdNZXJjb3N1cicpLCB2b2NhYj1Ob25lKToKICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybQogICAgICAgIHNlbGYuc3JfdHJhbnNmb3JtID0gc3JfdHJhbnNmb3JtCiAgICAgICAgc2VsZi5yZXR1cm5fc3IgPSByZXR1cm5fc3IKICAgICAgICBzZWxmLnJldHVybl9sYXlvdXQgPSByZXR1cm5fbGF5b3V0CiAgICAgICAgc2VsZi5sYXlvdXRzID0gbGlzdChsYXlvdXRzKQogICAgICAgIHNlbGYudm9jYWIgPSB2b2NhYgogICAgICAgIHNlbGYuc3BsaXQgPSBzcGxpdAogICAgICAgIHNlbGYuZGVncmFkYXRpb24gPSBzZWxmLmdldF9kZWdyYWRhdGlvbl90cmFuc2Zvcm1zKCkKCiAgICAgICAgaWYgc3BsaXQgaW4gWyd0cmFpbicsICd2YWwnXToKICAgICAgICAgICAgZGF0YV9wYXRoID0gb3MucGF0aC5qb2luKHBhdGgsICd0cmFpbicpCiAgICAgICAgICAgIGFsbF90cmFja3MgPSBzZWxmLmdldF9wYXRoKGRhdGFfcGF0aCwgc3BsaXQ9J3RyYWluJykKICAgICAgICAgICAgdHJhaW5fdHJhY2tzLCB2YWxfdHJhY2tzID0gc2VsZi5zcGxpdF90cmFja3MoYWxsX3RyYWNrcywgdGVzdF9zaXplPXZhbF9zaXplLCByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlKQogICAgICAgICAgICBzZWxmLnRyYWNrcyA9IHRyYWluX3RyYWNrcyBpZiBzcGxpdCA9PSAndHJhaW4nIGVsc2UgdmFsX3RyYWNrcwogICAgICAgIGVsaWYgc3BsaXQgPT0gJ3Rlc3RfbGFiZWwnOgogICAgICAgICAgICBzZWxmLnRyYWNrcyA9IHNlbGYuZ2V0X3BhdGgob3MucGF0aC5qb2luKHBhdGgsICd0ZXN0X2xhYmVsJyksIHNwbGl0PSd0ZXN0X2xhYmVsJykKICAgICAgICBlbGlmIHNwbGl0ID09ICdibGluZF90ZXN0JzoKICAgICAgICAgICAgc2VsZi50cmFja3MgPSBzZWxmLmdldF9wYXRoKG9zLnBhdGguam9pbihwYXRoLCAndGVzdCcpLCBzcGxpdD0ndGVzdCcpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gc3BsaXQ6IHtzcGxpdH0iKQoKICAgICAgICBzZWxmLmVudHJpZXMgPSBzZWxmLmJ1aWxkX2VudHJpZXMoc2VsZi50cmFja3MpCgogICAgICAgIHByaW50KGYnU3BsaXQ6IHtzcGxpdH0nKQogICAgICAgIHByaW50KGYnTnVtYmVyIG9mIHRyYWNrczoge2xlbihzZWxmLnRyYWNrcyl9JykKICAgICAgICBwcmludChmJ051bWJlciBvZiBzYW1wbGVzIChlbnRyaWVzKToge2xlbihzZWxmLmVudHJpZXMpfScpCgogICAgZGVmIGdldF9wYXRoKHNlbGYsIHBhdGgsIHNwbGl0KToKICAgICAgICB0cmFja3MgPSB7fQogICAgICAgIHRyYWNrX3BhdGhzID0gW10KCiAgICAgICAgaWYgc3BsaXQgaW4gKCd0ZXN0JywgJ3Rlc3RfbGFiZWwnKToKICAgICAgICAgICAgZm9yIHRyYWNrX25hbWUgaW4gb3MubGlzdGRpcihwYXRoKToKICAgICAgICAgICAgICAgIHRyYWNrX3BhdGhzLmFwcGVuZChvcy5wYXRoLmpvaW4ocGF0aCwgdHJhY2tfbmFtZSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZm9yIHNjZW5hcmlvIGluIFsnU2NlbmFyaW8tQScsICdTY2VuYXJpby1CJ106CiAgICAgICAgICAgICAgICBmb3IgY291bnRyeSBpbiBbJ0JyYXppbGlhbicsICdNZXJjb3N1ciddOgogICAgICAgICAgICAgICAgICAgIGRhdGFfcGF0aCA9IG9zLnBhdGguam9pbihwYXRoLCBzY2VuYXJpbywgY291bnRyeSkKICAgICAgICAgICAgICAgICAgICBpZiBub3Qgb3MucGF0aC5pc2RpcihkYXRhX3BhdGgpOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGZvciB0cmFja19uYW1lIGluIG9zLmxpc3RkaXIoZGF0YV9wYXRoKToKICAgICAgICAgICAgICAgICAgICAgICAgdHJhY2tfcGF0aHMuYXBwZW5kKG9zLnBhdGguam9pbihkYXRhX3BhdGgsIHRyYWNrX25hbWUpKQoKICAgICAgICBmb3IgdHJhY2tfcGF0aCBpbiB0cmFja19wYXRoczoKICAgICAgICAgICAgdHJhY2tfaWQgPSBvcy5wYXRoLmJhc2VuYW1lKHRyYWNrX3BhdGgpCiAgICAgICAgICAgIHRyYWNrX2xyLCB0cmFja19ociA9IFtdLCBbXQogICAgICAgICAgICB0cmFja19sYWJlbCwgdHJhY2tfbGF5b3V0ID0gTm9uZSwgTm9uZQoKICAgICAgICAgICAgZm9yIGl0ZW0gaW4gb3MubGlzdGRpcih0cmFja19wYXRoKToKICAgICAgICAgICAgICAgIGl0ZW1fcGF0aCA9IG9zLnBhdGguam9pbih0cmFja19wYXRoLCBpdGVtKQogICAgICAgICAgICAgICAgaWYgaXRlbS5sb3dlcigpLnN0YXJ0c3dpdGgoJ2xyJyk6CiAgICAgICAgICAgICAgICAgICAgdHJhY2tfbHIuYXBwZW5kKGl0ZW1fcGF0aCkKICAgICAgICAgICAgICAgIGVsaWYgaXRlbS5sb3dlcigpLnN0YXJ0c3dpdGgoJ2hyJyk6CiAgICAgICAgICAgICAgICAgICAgdHJhY2tfaHIuYXBwZW5kKGl0ZW1fcGF0aCkKICAgICAgICAgICAgICAgIGVsaWYgaXRlbS5sb3dlcigpLmVuZHN3aXRoKCcuanNvbicpIGFuZCBzcGxpdCAhPSAndGVzdCc6CiAgICAgICAgICAgICAgICAgICAgd2l0aCBvcGVuKGl0ZW1fcGF0aCwgJ3InLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgICAgICAgICAgICAgICAgICBhbm5vdGF0aW9ucyA9IGpzb24ubG9hZChmKQogICAgICAgICAgICAgICAgICAgIHRyYWNrX2xhYmVsID0gYW5ub3RhdGlvbnMuZ2V0KCdwbGF0ZV90ZXh0Jykuc3RyaXAoKQogICAgICAgICAgICAgICAgICAgIHRyYWNrX2xheW91dCA9IGFubm90YXRpb25zLmdldCgncGxhdGVfbGF5b3V0JykKCiAgICAgICAgICAgIHRyYWNrc1t0cmFja19pZF0gPSB7CiAgICAgICAgICAgICAgICAndHJhY2tfcGF0aCc6IHRyYWNrX3BhdGgsCiAgICAgICAgICAgICAgICAnbHJfcGF0aHMnOiBzb3J0ZWQodHJhY2tfbHIpLAogICAgICAgICAgICAgICAgJ2hyX3BhdGhzJzogc29ydGVkKHRyYWNrX2hyKSwKICAgICAgICAgICAgICAgICdsYWJlbCc6IHRyYWNrX2xhYmVsLAogICAgICAgICAgICAgICAgJ2xheW91dCc6IHRyYWNrX2xheW91dCwKICAgICAgICAgICAgfQoKICAgICAgICByZXR1cm4gdHJhY2tzCgogICAgZGVmIHNwbGl0X3RyYWNrcyhzZWxmLCB0cmFja3MsIHRlc3Rfc2l6ZT0wLjEsIHJhbmRvbV9zdGF0ZT00Mik6CiAgICAgICAgdHJhY2tfaWRzID0gc29ydGVkKHRyYWNrcy5rZXlzKCkpCiAgICAgICAgdHJhaW5faWRzLCB2YWxfaWRzID0gdHJhaW5fdGVzdF9zcGxpdCh0cmFja19pZHMsIHRlc3Rfc2l6ZT10ZXN0X3NpemUsIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUsIHNodWZmbGU9VHJ1ZSkKICAgICAgICB0cmFpbl90cmFja3MgPSB7dGlkOiB0cmFja3NbdGlkXSBmb3IgdGlkIGluIHRyYWluX2lkc30KICAgICAgICB2YWxfdHJhY2tzID0ge3RpZDogdHJhY2tzW3RpZF0gZm9yIHRpZCBpbiB2YWxfaWRzfQogICAgICAgIHJldHVybiB0cmFpbl90cmFja3MsIHZhbF90cmFja3MKCiAgICBkZWYgYnVpbGRfZW50cmllcyhzZWxmLCB0cmFja3MpOgogICAgICAgIGVudHJpZXMgPSBbXQogICAgICAgIGZvciB0cmFja19pZCwgdHJhY2sgaW4gdHJhY2tzLml0ZW1zKCk6CiAgICAgICAgICAgICMgRW50cnkgMTogb3JpZ2luYWwgTFIKICAgICAgICAgICAgZW50cmllcy5hcHBlbmQoewogICAgICAgICAgICAgICAgJ2VudHJ5X2lkJzogdHJhY2tfaWQgKyAnX2xyJywKICAgICAgICAgICAgICAgICd0cmFja19pZCc6IHRyYWNrX2lkLAogICAgICAgICAgICAgICAgJ3BhdGhzJzogdHJhY2tbJ2xyX3BhdGhzJ10sCiAgICAgICAgICAgICAgICAnaHJfcGF0aHMnOiB0cmFja1snaHJfcGF0aHMnXSwKICAgICAgICAgICAgICAgICd1c2VfaHInOiBGYWxzZSwKICAgICAgICAgICAgICAgICdsYWJlbCc6IHRyYWNrWydsYWJlbCddLAogICAgICAgICAgICAgICAgJ2xheW91dCc6IHRyYWNrWydsYXlvdXQnXSwKICAgICAgICAgICAgfSkKICAgICAgICAgICAgIyBFbnRyeSAyOiBIUi1kZWdyYWRlZCBwc2V1ZG8tTFIgKHRyYWluIG9ubHkpCiAgICAgICAgICAgIGlmIHNlbGYuc3BsaXQgPT0gJ3RyYWluJyBhbmQgbGVuKHRyYWNrWydocl9wYXRocyddKSA+IDA6CiAgICAgICAgICAgICAgICBlbnRyaWVzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAgICAgJ2VudHJ5X2lkJzogdHJhY2tfaWQgKyAnX2hyJywKICAgICAgICAgICAgICAgICAgICAndHJhY2tfaWQnOiB0cmFja19pZCwKICAgICAgICAgICAgICAgICAgICAncGF0aHMnOiB0cmFja1snaHJfcGF0aHMnXSwKICAgICAgICAgICAgICAgICAgICAnaHJfcGF0aHMnOiB0cmFja1snaHJfcGF0aHMnXSwKICAgICAgICAgICAgICAgICAgICAndXNlX2hyJzogVHJ1ZSwKICAgICAgICAgICAgICAgICAgICAnbGFiZWwnOiB0cmFja1snbGFiZWwnXSwKICAgICAgICAgICAgICAgICAgICAnbGF5b3V0JzogdHJhY2tbJ2xheW91dCddLAogICAgICAgICAgICAgICAgfSkKICAgICAgICByZXR1cm4gZW50cmllcwoKICAgIGRlZiBnZXRfZGVncmFkYXRpb25fdHJhbnNmb3JtcyhzZWxmKToKICAgICAgICByZXR1cm4gQS5Db21wb3NlKFsKICAgICAgICAgICAgQS5PbmVPZihbCiAgICAgICAgICAgICAgICBBLkdhdXNzaWFuQmx1cihibHVyX2xpbWl0PSgzLCA1KSwgcD0xLjApLAogICAgICAgICAgICAgICAgQS5Nb3Rpb25CbHVyKGJsdXJfbGltaXQ9KDMsIDUpLCBwPTEuMCksCiAgICAgICAgICAgIF0sIHA9MC43KSwKICAgICAgICAgICAgQS5PbmVPZihbCiAgICAgICAgICAgICAgICBBLkdhdXNzTm9pc2Uobm9pc2Vfc2NhbGVfZmFjdG9yPTAuMSwgcD0xLjApLAogICAgICAgICAgICAgICAgQS5NdWx0aXBsaWNhdGl2ZU5vaXNlKG11bHRpcGxpZXI9KDAuOSwgMS4xKSwgcD0xLjApLAogICAgICAgICAgICBdLCBwPTAuNyksCiAgICAgICAgICAgIEEuSW1hZ2VDb21wcmVzc2lvbihxdWFsaXR5X3JhbmdlPSgyMCwgNTApLCBwPTAuNSksCiAgICAgICAgICAgIEEuRG93bnNjYWxlKHNjYWxlX3JhbmdlPSgwLjMsIDAuNSksIHA9MC41KSwKICAgICAgICBdKQoKICAgIGRlZiBfbGF5b3V0X2luZGV4KHNlbGYsIGxheW91dCk6CiAgICAgICAgaWYgbGF5b3V0IGluIHNlbGYubGF5b3V0czoKICAgICAgICAgICAgcmV0dXJuIHNlbGYubGF5b3V0cy5pbmRleChsYXlvdXQpCiAgICAgICAgcmV0dXJuIC0xICAjIHVua25vd24gKGJsaW5kIHRlc3QpCgogICAgZGVmIF9fbGVuX18oc2VsZik6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLmVudHJpZXMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeCk6CiAgICAgICAgZW50cnkgPSBzZWxmLmVudHJpZXNbaWR4XQoKICAgICAgICBpbWFnZXMgPSBbXQogICAgICAgIHJhd19pbWFnZXMgPSBbXQogICAgICAgIGZvciBpbWdfcGF0aCBpbiBlbnRyeVsncGF0aHMnXToKICAgICAgICAgICAgaW1nID0gbnAuYXJyYXkoSW1hZ2Uub3BlbihpbWdfcGF0aCkuY29udmVydCgnUkdCJykpCiAgICAgICAgICAgIGlmIGVudHJ5Wyd1c2VfaHInXToKICAgICAgICAgICAgICAgIGltZyA9IHNlbGYuZGVncmFkYXRpb24oaW1hZ2U9aW1nKVsnaW1hZ2UnXSAgIyBIUiAtPiBwc2V1ZG8tTFIKICAgICAgICAgICAgcmF3X2ltYWdlcy5hcHBlbmQoaW1nKQogICAgICAgICAgICBpZiBzZWxmLnRyYW5zZm9ybSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGltZyA9IHNlbGYudHJhbnNmb3JtKGltYWdlPWltZylbJ2ltYWdlJ10KICAgICAgICAgICAgaW1hZ2VzLmFwcGVuZChpbWcpCgogICAgICAgIHNhbXBsZSA9IHsKICAgICAgICAgICAgJ3RyYWNrX2lkJzogZW50cnlbJ3RyYWNrX2lkJ10sCiAgICAgICAgICAgICdpbWFnZXMnOiBpbWFnZXMsCiAgICAgICAgICAgICdsYWJlbCc6IGVudHJ5WydsYWJlbCddLAogICAgICAgICAgICAnbGF5b3V0Jzogc2VsZi5fbGF5b3V0X2luZGV4KGVudHJ5WydsYXlvdXQnXSksCiAgICAgICAgfQoKICAgICAgICBpZiBzZWxmLnJldHVybl9zciBhbmQgc2VsZi5zcl90cmFuc2Zvcm0gaXMgbm90IE5vbmUgYW5kIGxlbihlbnRyeVsnaHJfcGF0aHMnXSkgPiAwOgogICAgICAgICAgICBzcl90YXJnZXRzID0gW10KICAgICAgICAgICAgZm9yIGhyX3BhdGggaW4gZW50cnlbJ2hyX3BhdGhzJ106CiAgICAgICAgICAgICAgICBociA9IG5wLmFycmF5KEltYWdlLm9wZW4oaHJfcGF0aCkuY29udmVydCgnUkdCJykpCiAgICAgICAgICAgICAgICBzcl90YXJnZXRzLmFwcGVuZChzZWxmLnNyX3RyYW5zZm9ybShpbWFnZT1ocilbJ2ltYWdlJ10pCiAgICAgICAgICAgIHNhbXBsZVsnc3JfdGFyZ2V0J10gPSBzcl90YXJnZXRzCgogICAgICAgIHJldHVybiBzYW1wbGUK",
"datasets/transforms.py": "IiIiQWxidW1lbnRhdGlvbnMgcGlwZWxpbmVzIHNoYXJlZCBieSBhbGwgYmFzZWxpbmVzLgoKTm9ybWFsaXphdGlvbiBpcyBmaXhlZCBhdCBtZWFuPXN0ZD0wLjUgKG1hdGNoaW5nIHRoZSBvcmlnaW5hbCB0cmFpbmluZyBub3RlYm9va3MpLgpUaGUgU1ItdGFyZ2V0IHRyYW5zZm9ybSBvbmx5IHJlc2l6ZXMgKyBub3JtYWxpemVzIHRoZSBjbGVhbiBIUiBmcmFtZXMgc28gdGhleSBjYW4Kc3VwZXJ2aXNlIHRoZSBTUiBicmFuY2ggYXQgYGBzY2FsZWBgIHggdGhlIE9DUiBpbnB1dCBzaXplLgoiIiIKCmltcG9ydCBhbGJ1bWVudGF0aW9ucyBhcyBBCmZyb20gYWxidW1lbnRhdGlvbnMucHl0b3JjaCBpbXBvcnQgVG9UZW5zb3JWMgoKTk9STV9NRUFOID0gKDAuNSwgMC41LCAwLjUpCk5PUk1fU1REID0gKDAuNSwgMC41LCAwLjUpCgoKZGVmIGJ1aWxkX3RyYW5zZm9ybXMoaW1nX0g9MzIsIGltZ19XPTEyOCk6CiAgICB0cmFpbl90cmFuc2Zvcm1zID0gQS5Db21wb3NlKFsKICAgICAgICBBLlJlc2l6ZShoZWlnaHQ9aW1nX0gsIHdpZHRoPWltZ19XKSwKICAgICAgICBBLkFmZmluZShzY2FsZT0oMC45NSwgMS4wNSksIHRyYW5zbGF0ZV9wZXJjZW50PSgwLjA1LCAwLjA1KSwgcm90YXRlPSgtNSwgNSksIGZpbGw9MTI4LCBwPTAuNSksCiAgICAgICAgQS5QZXJzcGVjdGl2ZShzY2FsZT0oMC4wMiwgMC4wNSksIHA9MC4zKSwKICAgICAgICBBLlJhbmRvbUJyaWdodG5lc3NDb250cmFzdChicmlnaHRuZXNzX2xpbWl0PTAuMTUsIGNvbnRyYXN0X2xpbWl0PTAuMTUsIHA9MC40KSwKICAgICAgICBBLkh1ZVNhdHVyYXRpb25WYWx1ZShodWVfc2hpZnRfbGltaXQ9MTAsIHNhdF9zaGlmdF9saW1pdD0yMCwgdmFsX3NoaWZ0X2xpbWl0PTIwLCBwPTAuMyksCiAgICAgICAgQS5DaGFubmVsU2h1ZmZsZShwPTAuMyksCiAgICAgICAgQS5Db2Fyc2VEcm9wb3V0KG51bV9ob2xlc19yYW5nZT0oMiwgNSksIGhvbGVfaGVpZ2h0X3JhbmdlPSg0LCA4KSwgaG9sZV93aWR0aF9yYW5nZT0oNCwgOCksIHA9MC4zKSwKICAgICAgICBBLk5vcm1hbGl6ZShtZWFuPU5PUk1fTUVBTiwgc3RkPU5PUk1fU1REKSwKICAgICAgICBUb1RlbnNvclYyKCksCiAgICBdKQoKICAgIHZhbF90ZXN0X3RyYW5zZm9ybXMgPSBBLkNvbXBvc2UoWwogICAgICAgIEEuUmVzaXplKGhlaWdodD1pbWdfSCwgd2lkdGg9aW1nX1cpLAogICAgICAgIEEuTm9ybWFsaXplKG1lYW49Tk9STV9NRUFOLCBzdGQ9Tk9STV9TVEQpLAogICAgICAgIFRvVGVuc29yVjIoKSwKICAgIF0pCgogICAgcmV0dXJuIHRyYWluX3RyYW5zZm9ybXMsIHZhbF90ZXN0X3RyYW5zZm9ybXMKCgpkZWYgYnVpbGRfc3JfdGFyZ2V0X3RyYW5zZm9ybShpbWdfSD0zMiwgaW1nX1c9MTI4LCBzY2FsZT0yKToKICAgICIiIkNsZWFuLUhSIHRhcmdldCBmb3IgdGhlIFNSIGJyYW5jaCAobm8gYXVnbWVudGF0aW9uLCBqdXN0IHJlc2l6ZStub3JtYWxpemUpLiIiIgogICAgcmV0dXJuIEEuQ29tcG9zZShbCiAgICAgICAgQS5SZXNpemUoaGVpZ2h0PWltZ19IICogc2NhbGUsIHdpZHRoPWltZ19XICogc2NhbGUpLAogICAgICAgIEEuTm9ybWFsaXplKG1lYW49Tk9STV9NRUFOLCBzdGQ9Tk9STV9TVEQpLAogICAgICAgIFRvVGVuc29yVjIoKSwKICAgIF0pCg==",
"trainers/__init__.py": "ZnJvbSAudHJhaW5lciBpbXBvcnQgYnVpbGRfdHJhaW5pbmdfY29tcG9uZW50cywgY29tcHV0ZV9hY2N1cmFjeSwgcnVuX3RyYWluaW5nLCB0cmFpbl9tb2RlbA0KZnJvbSAuaGFybmVzcyBpbXBvcnQgdHJhaW5faGFybmVzcw0KDQpfX2FsbF9fID0gWw0KICAgICJjb21wdXRlX2FjY3VyYWN5IiwNCiAgICAidHJhaW5fbW9kZWwiLA0KICAgICJidWlsZF90cmFpbmluZ19jb21wb25lbnRzIiwNCiAgICAicnVuX3RyYWluaW5nIiwNCiAgICAidHJhaW5faGFybmVzcyIsDQpdDQo=",
"trainers/harness.py": "IiIiVW5pZmllZCB0cmFpbmluZyBoYXJuZXNzIHdpdGggb3B0aW9uYWwgU1IgKyBsYXlvdXQgYXV4aWxpYXJ5IGxvc3Nlcy4KClRvdGFsIGxvc3MgPSBDRShvY3IpICsgc3JfbG9zc193ZWlnaHQgKiBMMShzciwgSFIpICsgbGF5b3V0X2xvc3Nfd2VpZ2h0ICogQ0UobGF5b3V0KQoKV29ya3MgZm9yIGV2ZXJ5IHJlZ2lzdHJ5IGNvbmZpZ3VyYXRpb24uIFVzZXMgYGBjb2xsYXRlX2hhcm5lc3NgYCBiYXRjaGVzIChkaWN0cykKYW5kIHRoZSBtb2RlbCdzIGBgcmV0dXJuX2F1eD1UcnVlYGAgcGF0aC4gVGhlIGxlZ2FjeSBgYHRyYWluX21vZGVsYGAgaW4KYGB0cmFpbmVyLnB5YGAgaXMgbGVmdCB1bnRvdWNoZWQgZm9yIHRoZSBvcmlnaW5hbCBzaW5nbGUtbG9zcyBiYXNlbGluZXMuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRxZG0uYXV0byBpbXBvcnQgdHFkbQoKCmRlZiBjb21wdXRlX2FjY3VyYWN5KGxvZ2l0cywgdGFyZ2V0cyk6CiAgICBwcmVkcyA9IGxvZ2l0cy5hcmdtYXgoZGltPTIpCiAgICBjb3JyZWN0ID0gKHByZWRzID09IHRhcmdldHMpLmFsbChkaW09MSkuc3VtKCkuaXRlbSgpCiAgICByZXR1cm4gY29ycmVjdCAvIHRhcmdldHMuc2l6ZSgwKQoKCmRlZiBfc3JfbG9zcyhzcl9vdXQsIHNyX3RhcmdldCwgYmF0Y2hfc2l6ZSwgc3JfY3JpdGVyaW9uKToKICAgICIiIkFsaWduIFNSIG91dHB1dCAoQipGX3VzZWQsIEMsIEhzLCBXcykgd2l0aCB0YXJnZXQgKEIsIEYsIEMsIEhzLCBXcykuIiIiCiAgICBmX3VzZWQgPSBzcl9vdXQuc2l6ZSgwKSAvLyBiYXRjaF9zaXplCiAgICBmX3RvdGFsID0gc3JfdGFyZ2V0LnNpemUoMSkKICAgIGlmIGZfdXNlZCA9PSBmX3RvdGFsOgogICAgICAgIHRhcmdldCA9IHNyX3RhcmdldC5yZXNoYXBlKC0xLCAqc3JfdGFyZ2V0LnNoYXBlWzI6XSkKICAgIGVsc2U6ICAjIHNpbmdsZS1mcmFtZSBtb2RlbCB1c2VkIHRoZSBtaWRkbGUgZnJhbWUKICAgICAgICBtaWQgPSBmX3RvdGFsIC8vIDIKICAgICAgICB0YXJnZXQgPSBzcl90YXJnZXRbOiwgbWlkOm1pZCArIGZfdXNlZF0ucmVzaGFwZSgtMSwgKnNyX3RhcmdldC5zaGFwZVsyOl0pCiAgICByZXR1cm4gc3JfY3JpdGVyaW9uKHNyX291dCwgdGFyZ2V0KQoKCmRlZiB0cmFpbl9oYXJuZXNzKG1vZGVsLCBjZmcsIHRyYWluX2xvYWRlciwgdmFsX2xvYWRlciwgYmVzdF9tb2RlbF9wYXRoLCBkZXZpY2U9ImNwdSIpOgogICAgY3JpdGVyaW9uID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICBzcl9jcml0ZXJpb24gPSBubi5MMUxvc3MoKQogICAgbGF5b3V0X2NyaXRlcmlvbiA9IG5uLkNyb3NzRW50cm9weUxvc3MoaWdub3JlX2luZGV4PS0xKQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbShtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWNmZy5scikKICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5Db3NpbmVBbm5lYWxpbmdMUihvcHRpbWl6ZXIsIFRfbWF4PWNmZy5lcG9jaHMsIGV0YV9taW49MWUtNikKCiAgICB1c2VfYW1wID0gYm9vbChnZXRhdHRyKGNmZywgInVzZV9hbXAiLCBGYWxzZSkpIGFuZCBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCJjdWRhIikKICAgIHNjYWxlciA9IHRvcmNoLmN1ZGEuYW1wLkdyYWRTY2FsZXIoZW5hYmxlZD11c2VfYW1wKQoKICAgIGJlc3RfdmFsX2FjYyA9IDAuMAogICAgYnJlYWtfY291bnQgPSAwCiAgICBoaXN0b3J5ID0geyJ0cmFpbl9sb3NzIjogW10sICJ2YWxfbG9zcyI6IFtdLCAidHJhaW5fYWNjIjogW10sICJ2YWxfYWNjIjogW119CgogICAgd2FybXVwID0gY2ZnLndhcm11cF9lcG9jaHMKICAgIGlmIHdhcm11cCA+IDA6CiAgICAgICAgZm9yIHAgaW4gbW9kZWwuZXh0cmFjdG9yLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gRmFsc2UKCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoY2ZnLmVwb2Nocyk6CiAgICAgICAgaWYgd2FybXVwID4gMCBhbmQgZXBvY2ggPT0gd2FybXVwOgogICAgICAgICAgICBmb3IgcCBpbiBtb2RlbC5leHRyYWN0b3IucGFyYW1ldGVycygpOgogICAgICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gVHJ1ZQoKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgaWYgd2FybXVwID4gMCBhbmQgZXBvY2ggPCB3YXJtdXA6CiAgICAgICAgICAgIG1vZGVsLmV4dHJhY3Rvci5ldmFsKCkKCiAgICAgICAgZXBfbG9zcywgZXBfYWNjID0gMC4wLCAwLjAKICAgICAgICBmb3IgYmF0Y2ggaW4gdHFkbSh0cmFpbl9sb2FkZXIsIGRlc2M9ZiJUcmFpbiBle2Vwb2NofSIsIGxlYXZlPUZhbHNlKToKICAgICAgICAgICAgaW1hZ2VzID0gYmF0Y2hbImltYWdlcyJdLnRvKGRldmljZSkKICAgICAgICAgICAgdGFyZ2V0cyA9IGJhdGNoWyJ0YXJnZXRzIl0udG8oZGV2aWNlKQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKCiAgICAgICAgICAgIHdpdGggdG9yY2guY3VkYS5hbXAuYXV0b2Nhc3QoZW5hYmxlZD11c2VfYW1wKToKICAgICAgICAgICAgICAgIG91dCA9IG1vZGVsKGltYWdlcywgcmV0dXJuX2F1eD1UcnVlKQogICAgICAgICAgICAgICAgbG9naXRzID0gb3V0WyJsb2dpdHMiXQogICAgICAgICAgICAgICAgbG9zcyA9IGNyaXRlcmlvbihsb2dpdHMucGVybXV0ZSgwLCAyLCAxKSwgdGFyZ2V0cykKCiAgICAgICAgICAgICAgICBpZiBjZmcudXNlX3NyIGFuZCBvdXRbInNyIl0gaXMgbm90IE5vbmUgYW5kICJzcl90YXJnZXQiIGluIGJhdGNoOgogICAgICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzICsgY2ZnLnNyX2xvc3Nfd2VpZ2h0ICogX3NyX2xvc3MoCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFsic3IiXSwgYmF0Y2hbInNyX3RhcmdldCJdLnRvKGRldmljZSksIGltYWdlcy5zaXplKDApLCBzcl9jcml0ZXJpb24KICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBpZiBjZmcudXNlX2xheW91dF9oZWFkIGFuZCBvdXRbImxheW91dCJdIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzICsgY2ZnLmxheW91dF9sb3NzX3dlaWdodCAqIGxheW91dF9jcml0ZXJpb24oCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFsibGF5b3V0Il0sIGJhdGNoWyJsYXlvdXQiXS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgc2NhbGVyLnNjYWxlKGxvc3MpLmJhY2t3YXJkKCkKICAgICAgICAgICAgc2NhbGVyLnVuc2NhbGVfKG9wdGltaXplcikKICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgbWF4X25vcm09NS4wKQogICAgICAgICAgICBzY2FsZXIuc3RlcChvcHRpbWl6ZXIpCiAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICBlcF9sb3NzICs9IGxvc3MuaXRlbSgpCiAgICAgICAgICAgIGVwX2FjYyArPSBjb21wdXRlX2FjY3VyYWN5KGxvZ2l0cy5kZXRhY2goKS5jcHUoKSwgdGFyZ2V0cy5jcHUoKSkKCiAgICAgICAgaGlzdG9yeVsidHJhaW5fbG9zcyJdLmFwcGVuZChlcF9sb3NzIC8gbGVuKHRyYWluX2xvYWRlcikpCiAgICAgICAgaGlzdG9yeVsidHJhaW5fYWNjIl0uYXBwZW5kKGVwX2FjYyAvIGxlbih0cmFpbl9sb2FkZXIpKQoKICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICBlcF9sb3NzLCBlcF9hY2MgPSAwLjAsIDAuMAogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBmb3IgYmF0Y2ggaW4gdHFkbSh2YWxfbG9hZGVyLCBkZXNjPWYiVmFsIGV7ZXBvY2h9IiwgbGVhdmU9RmFsc2UpOgogICAgICAgICAgICAgICAgaW1hZ2VzID0gYmF0Y2hbImltYWdlcyJdLnRvKGRldmljZSkKICAgICAgICAgICAgICAgIHRhcmdldHMgPSBiYXRjaFsidGFyZ2V0cyJdLnRvKGRldmljZSkKICAgICAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGltYWdlcykKICAgICAgICAgICAgICAgIGVwX2xvc3MgKz0gY3JpdGVyaW9uKGxvZ2l0cy5wZXJtdXRlKDAsIDIsIDEpLCB0YXJnZXRzKS5pdGVtKCkKICAgICAgICAgICAgICAgIGVwX2FjYyArPSBjb21wdXRlX2FjY3VyYWN5KGxvZ2l0cy5jcHUoKSwgdGFyZ2V0cy5jcHUoKSkKCiAgICAgICAgaGlzdG9yeVsidmFsX2xvc3MiXS5hcHBlbmQoZXBfbG9zcyAvIGxlbih2YWxfbG9hZGVyKSkKICAgICAgICBoaXN0b3J5WyJ2YWxfYWNjIl0uYXBwZW5kKGVwX2FjYyAvIGxlbih2YWxfbG9hZGVyKSkKICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCgogICAgICAgIHByaW50KGYifCBFcG9jaCB7ZXBvY2g6M2R9IHwgVHJhaW4gTG9zcyB7aGlzdG9yeVsndHJhaW5fbG9zcyddWy0xXTouNGZ9ICIKICAgICAgICAgICAgICBmIkFjYyB7aGlzdG9yeVsndHJhaW5fYWNjJ11bLTFdOi40Zn0gfCBWYWwgTG9zcyB7aGlzdG9yeVsndmFsX2xvc3MnXVstMV06LjRmfSAiCiAgICAgICAgICAgICAgZiJBY2Mge2hpc3RvcnlbJ3ZhbF9hY2MnXVstMV06LjRmfSB8IikKCiAgICAgICAgaWYgaGlzdG9yeVsidmFsX2FjYyJdWy0xXSA+IGJlc3RfdmFsX2FjYzoKICAgICAgICAgICAgYnJlYWtfY291bnQgPSAwCiAgICAgICAgICAgIGJlc3RfdmFsX2FjYyA9IGhpc3RvcnlbInZhbF9hY2MiXVstMV0KICAgICAgICAgICAgdG9yY2guc2F2ZSh7ImVwb2NoIjogZXBvY2gsICJtb2RlbF9zdGF0ZV9kaWN0IjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAiYmVzdF92YWxfYWNjIjogYmVzdF92YWxfYWNjLCAiaGlzdG9yeSI6IGhpc3RvcnksCiAgICAgICAgICAgICAgICAgICAgICAgICJjb25maWciOiB2YXJzKGNmZyl9LCBiZXN0X21vZGVsX3BhdGgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYnJlYWtfY291bnQgKz0gMQogICAgICAgIGlmIGJyZWFrX2NvdW50ID49IGNmZy5lYXJseV9zdG9wX2NvdW50OgogICAgICAgICAgICBwcmludChmIkVhcmx5IHN0b3AgYXQgZXBvY2gge2Vwb2NofSB8IGJlc3QgdmFsIGFjYyB7YmVzdF92YWxfYWNjOi40Zn0iKQogICAgICAgICAgICBicmVhawoKICAgIHJldHVybiBtb2RlbCwgaGlzdG9yeQo=",
"trainers/trainer.py": "aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtDQoNCmZyb20gY29uZmlnIGltcG9ydCBDb25maWcNCmZyb20gdmlzdWFsaXphdGlvbiBpbXBvcnQgdmlzdWFsaXplX3ZhbF9zYW1wbGVzDQoNCg0KZGVmIGNvbXB1dGVfYWNjdXJhY3kobG9naXRzOiB0b3JjaC5UZW5zb3IsIHRhcmdldHM6IHRvcmNoLlRlbnNvcikgLT4gZmxvYXQ6DQogICAgcHJlZHMgPSBsb2dpdHMuYXJnbWF4KGRpbT0yKQ0KICAgIGNvcnJlY3QgPSAocHJlZHMgPT0gdGFyZ2V0cykuYWxsKGRpbT0xKS5zdW0oKS5pdGVtKCkNCiAgICByZXR1cm4gY29ycmVjdCAvIHRhcmdldHMuc2l6ZSgwKQ0KDQoNCmRlZiB0cmFpbl9tb2RlbCgNCiAgICBtb2RlbCwNCiAgICBvcHRpbWl6ZXIsDQogICAgc2NoZWR1bGVyLA0KICAgIGNyaXRlcmlvbiwNCiAgICB0cmFpbl9sb2FkZXIsDQogICAgdmFsX2xvYWRlciwNCiAgICBlcG9jaHMsDQogICAgZWFybHlfc3RvcF9jb3VudCwNCiAgICBiZXN0X21vZGVsX3BhdGgsDQogICAgbG9nX2ludGVydmFsLA0KICAgIHZvY2FiLA0KICAgIHdhcm11cF9lcG9jaHM9MCwNCiAgICBkZXZpY2U9ImNwdSIsDQopOg0KICAgIGJlc3RfdmFsX2FjYyA9IDAuMA0KICAgIGJyZWFrX2NvdW50ID0gMA0KICAgIHRyYWluX2xvc3NfaGlzdCwgdmFsX2xvc3NfaGlzdCA9IFtdLCBbXQ0KICAgIHRyYWluX2FjY19oaXN0LCB2YWxfYWNjX2hpc3QgPSBbXSwgW10NCg0KICAgIGJhc2VfbW9kZWwgPSBtb2RlbC5tb2R1bGUgaWYgaXNpbnN0YW5jZShtb2RlbCwgbm4uRGF0YVBhcmFsbGVsKSBlbHNlIG1vZGVsDQoNCiAgICBpZiB3YXJtdXBfZXBvY2hzID4gMDoNCiAgICAgICAgZm9yIHAgaW4gYmFzZV9tb2RlbC5leHRyYWN0b3IucGFyYW1ldGVycygpOg0KICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gRmFsc2UNCiAgICAgICAgcHJpbnQoZiJXYXJtdXA6IGZyZWV6ZSBleHRyYWN0b3IgZm9yIGZpcnN0IHt3YXJtdXBfZXBvY2hzfSBlcG9jaHMiKQ0KDQogICAgZm9yIGVwb2NoIGluIHRxZG0ocmFuZ2UoZXBvY2hzKSwgZGVzYz0iRXBvY2giLCBwb3NpdGlvbj0wKToNCiAgICAgICAgaWYgd2FybXVwX2Vwb2NocyA+IDAgYW5kIGVwb2NoID09IHdhcm11cF9lcG9jaHM6DQogICAgICAgICAgICBmb3IgcCBpbiBiYXNlX21vZGVsLmV4dHJhY3Rvci5wYXJhbWV0ZXJzKCk6DQogICAgICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkID0gVHJ1ZQ0KICAgICAgICAgICAgcHJpbnQoZiJVbmZyZWV6ZSBleHRyYWN0b3IgYXQgZXBvY2gge2Vwb2NofSIpDQoNCiAgICAgICAgbW9kZWwudHJhaW4oKQ0KICAgICAgICBpZiB3YXJtdXBfZXBvY2hzID4gMCBhbmQgZXBvY2ggPCB3YXJtdXBfZXBvY2hzOg0KICAgICAgICAgICAgYmFzZV9tb2RlbC5leHRyYWN0b3IuZXZhbCgpDQoNCiAgICAgICAgZXBvY2hfbG9zcyA9IDAuMA0KICAgICAgICBlcG9jaF9hY2MgPSAwLjANCiAgICAgICAgZm9yIGltYWdlcywgdGFyZ2V0cyBpbiB0cWRtKHRyYWluX2xvYWRlciwgZGVzYz0iVHJhaW4iLCBwb3NpdGlvbj0xLCBsZWF2ZT1GYWxzZSk6DQogICAgICAgICAgICBpbWFnZXMgPSBpbWFnZXMudG8oZGV2aWNlKQ0KICAgICAgICAgICAgdGFyZ2V0cyA9IHRhcmdldHMudG8oZGV2aWNlKQ0KDQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkNCiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGltYWdlcykNCiAgICAgICAgICAgIGxvc3MgPSBjcml0ZXJpb24obG9naXRzLnBlcm11dGUoMCwgMiwgMSksIHRhcmdldHMpDQogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkNCiAgICAgICAgICAgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIG1heF9ub3JtPTUuMCkNCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkNCg0KICAgICAgICAgICAgZXBvY2hfbG9zcyArPSBsb3NzLml0ZW0oKQ0KICAgICAgICAgICAgZXBvY2hfYWNjICs9IGNvbXB1dGVfYWNjdXJhY3kobG9naXRzLmRldGFjaCgpLmNwdSgpLCB0YXJnZXRzLmNwdSgpKQ0KDQogICAgICAgIHRyYWluX2xvc3NfaGlzdC5hcHBlbmQoZXBvY2hfbG9zcyAvIGxlbih0cmFpbl9sb2FkZXIpKQ0KICAgICAgICB0cmFpbl9hY2NfaGlzdC5hcHBlbmQoZXBvY2hfYWNjIC8gbGVuKHRyYWluX2xvYWRlcikpDQoNCiAgICAgICAgbW9kZWwuZXZhbCgpDQogICAgICAgIGVwb2NoX2xvc3MgPSAwLjANCiAgICAgICAgZXBvY2hfYWNjID0gMC4wDQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgZm9yIGltYWdlcywgdGFyZ2V0cyBpbiB0cWRtKHZhbF9sb2FkZXIsIGRlc2M9IlZhbCIsIHBvc2l0aW9uPTEsIGxlYXZlPUZhbHNlKToNCiAgICAgICAgICAgICAgICBpbWFnZXMgPSBpbWFnZXMudG8oZGV2aWNlKQ0KICAgICAgICAgICAgICAgIHRhcmdldHMgPSB0YXJnZXRzLnRvKGRldmljZSkNCg0KICAgICAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGltYWdlcykNCiAgICAgICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cy5wZXJtdXRlKDAsIDIsIDEpLCB0YXJnZXRzKQ0KICAgICAgICAgICAgICAgIGVwb2NoX2xvc3MgKz0gbG9zcy5pdGVtKCkNCiAgICAgICAgICAgICAgICBlcG9jaF9hY2MgKz0gY29tcHV0ZV9hY2N1cmFjeShsb2dpdHMuY3B1KCksIHRhcmdldHMuY3B1KCkpDQoNCiAgICAgICAgdmFsX2xvc3NfaGlzdC5hcHBlbmQoZXBvY2hfbG9zcyAvIGxlbih2YWxfbG9hZGVyKSkNCiAgICAgICAgdmFsX2FjY19oaXN0LmFwcGVuZChlcG9jaF9hY2MgLyBsZW4odmFsX2xvYWRlcikpDQoNCiAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQ0KDQogICAgICAgIGlmIGVwb2NoICUgbG9nX2ludGVydmFsID09IDA6DQogICAgICAgICAgICBwcmludCgNCiAgICAgICAgICAgICAgICBmInwgRXBvY2gge2Vwb2NoOjNkfSAiDQogICAgICAgICAgICAgICAgZiJ8IFRyYWluIExvc3Mge3RyYWluX2xvc3NfaGlzdFstMV06LjRmfSAgQWNjIHt0cmFpbl9hY2NfaGlzdFstMV06LjRmfSAiDQogICAgICAgICAgICAgICAgZiJ8IFZhbCBMb3NzIHt2YWxfbG9zc19oaXN0Wy0xXTouNGZ9ICBBY2Mge3ZhbF9hY2NfaGlzdFstMV06LjRmfSB8Ig0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgdmlzdWFsaXplX3ZhbF9zYW1wbGVzKG1vZGVsLCB2YWxfbG9hZGVyLCB2b2NhYj12b2NhYiwgZGV2aWNlPWRldmljZSwgbl9zYW1wbGVzPTIpDQoNCiAgICAgICAgaWYgdmFsX2FjY19oaXN0Wy0xXSA+IGJlc3RfdmFsX2FjYzoNCiAgICAgICAgICAgIGJyZWFrX2NvdW50ID0gMA0KICAgICAgICAgICAgYmVzdF92YWxfYWNjID0gdmFsX2FjY19oaXN0Wy0xXQ0KICAgICAgICAgICAgdG9yY2guc2F2ZSgNCiAgICAgICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgICAgICJlcG9jaCI6IGVwb2NoLA0KICAgICAgICAgICAgICAgICAgICAibW9kZWxfc3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwNCiAgICAgICAgICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZV9kaWN0Ijogb3B0aW1pemVyLnN0YXRlX2RpY3QoKSwNCiAgICAgICAgICAgICAgICAgICAgInNjaGVkdWxlcl9zdGF0ZV9kaWN0Ijogc2NoZWR1bGVyLnN0YXRlX2RpY3QoKSwNCiAgICAgICAgICAgICAgICAgICAgImJlc3RfdmFsX2FjYyI6IHZhbF9hY2NfaGlzdFstMV0sDQogICAgICAgICAgICAgICAgICAgICJicmVha19jb3VudCI6IGJyZWFrX2NvdW50LA0KICAgICAgICAgICAgICAgICAgICAidHJhaW5fbG9zc19oaXN0IjogdHJhaW5fbG9zc19oaXN0LA0KICAgICAgICAgICAgICAgICAgICAidmFsX2xvc3NfaGlzdCI6IHZhbF9sb3NzX2hpc3QsDQogICAgICAgICAgICAgICAgICAgICJ0cmFpbl9hY2NfaGlzdCI6IHRyYWluX2FjY19oaXN0LA0KICAgICAgICAgICAgICAgICAgICAidmFsX2FjY19oaXN0IjogdmFsX2FjY19oaXN0LA0KICAgICAgICAgICAgICAgIH0sDQogICAgICAgICAgICAgICAgYmVzdF9tb2RlbF9wYXRoLA0KICAgICAgICAgICAgKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgYnJlYWtfY291bnQgKz0gMQ0KDQogICAgICAgIGlmIGJyZWFrX2NvdW50ID49IGVhcmx5X3N0b3BfY291bnQ6DQogICAgICAgICAgICBwcmludChmIkVhcmx5IHN0b3AgYXQgZXBvY2gge2Vwb2NofSB8IGJlc3QgdmFsIGFjYzoge2Jlc3RfdmFsX2FjYzouNGZ9IikNCiAgICAgICAgICAgIGJyZWFrDQoNCiAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChiZXN0X21vZGVsX3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UpDQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRbIm1vZGVsX3N0YXRlX2RpY3QiXSkNCiAgICBtb2RlbC5ldmFsKCkNCiAgICBwcmludCgiVHJhaW5pbmcgZG9uZSwgbG9hZGVkIGJlc3QgbW9kZWwiKQ0KDQogICAgcmV0dXJuIG1vZGVsLCB7DQogICAgICAgICJ0cmFpbl9sb3NzIjogdHJhaW5fbG9zc19oaXN0LA0KICAgICAgICAidmFsX2xvc3MiOiB2YWxfbG9zc19oaXN0LA0KICAgICAgICAidHJhaW5fYWNjIjogdHJhaW5fYWNjX2hpc3QsDQogICAgICAgICJ2YWxfYWNjIjogdmFsX2FjY19oaXN0LA0KICAgIH0NCg0KDQpkZWYgYnVpbGRfdHJhaW5pbmdfY29tcG9uZW50cyhtb2RlbCwgbHI6IGZsb2F0LCBlcG9jaHM6IGludCk6DQogICAgY3JpdGVyaW9uID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpDQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbShtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWxyKQ0KICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5Db3NpbmVBbm5lYWxpbmdMUihvcHRpbWl6ZXIsIFRfbWF4PWVwb2NocykNCiAgICByZXR1cm4gY3JpdGVyaW9uLCBvcHRpbWl6ZXIsIHNjaGVkdWxlcg0KDQoNCmRlZiBydW5fdHJhaW5pbmcobW9kZWwsIHRyYWluX2xvYWRlciwgdmFsX2xvYWRlciwgY2ZnKToNCiAgICBkZXZpY2UgPSBnZXRhdHRyKGNmZywgImRldmljZSIsICJjcHUiKQ0KICAgIG1vZGVsID0gbW9kZWwudG8oZGV2aWNlKQ0KDQogICAgY3JpdGVyaW9uLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciA9IGJ1aWxkX3RyYWluaW5nX2NvbXBvbmVudHMoDQogICAgICAgIG1vZGVsPW1vZGVsLA0KICAgICAgICBscj1jZmcubHIsDQogICAgICAgIGVwb2Nocz1jZmcuZXBvY2hzLA0KICAgICkNCg0KICAgIHRyYWluZWRfbW9kZWwsIGhpc3RvcnkgPSB0cmFpbl9tb2RlbCgNCiAgICAgICAgbW9kZWw9bW9kZWwsDQogICAgICAgIG9wdGltaXplcj1vcHRpbWl6ZXIsDQogICAgICAgIHNjaGVkdWxlcj1zY2hlZHVsZXIsDQogICAgICAgIGNyaXRlcmlvbj1jcml0ZXJpb24sDQogICAgICAgIHRyYWluX2xvYWRlcj10cmFpbl9sb2FkZXIsDQogICAgICAgIHZhbF9sb2FkZXI9dmFsX2xvYWRlciwNCiAgICAgICAgZXBvY2hzPWNmZy5lcG9jaHMsDQogICAgICAgIGVhcmx5X3N0b3BfY291bnQ9Y2ZnLmVhcmx5X3N0b3BfY291bnQsDQogICAgICAgIGJlc3RfbW9kZWxfcGF0aD1jZmcuYmVzdF9tb2RlbF9wYXRoLA0KICAgICAgICBsb2dfaW50ZXJ2YWw9Y2ZnLmxvZ19pbnRlcnZhbCwNCiAgICAgICAgdm9jYWI9Y2ZnLnZvY2FiLA0KICAgICAgICB3YXJtdXBfZXBvY2hzPWNmZy53YXJtdXBfZXBvY2hzLA0KICAgICAgICBkZXZpY2U9ZGV2aWNlLA0KICAgICkNCiAgICByZXR1cm4gdHJhaW5lZF9tb2RlbCwgaGlzdG9yeQ0K",
"evaluation/__init__.py": "ZnJvbSAuZXZhbF9oYXJuZXNzIGltcG9ydCBldmFsdWF0ZV9tb2RlbCwgcnVuX2NvbXBhcmlzb24sIGZvcm1hdF90YWJsZQoKX19hbGxfXyA9IFsiZXZhbHVhdGVfbW9kZWwiLCAicnVuX2NvbXBhcmlzb24iLCAiZm9ybWF0X3RhYmxlIl0K",
"evaluation/eval_harness.py": "IiIiVW5pZmllZCBldmFsdWF0aW9uIGhhcm5lc3Mgb24gdGhlIGxhYmVsbGVkIHRlc3Qgc2V0IChgYHRlc3RfbGFiZWxgYCkuCgpSZXBvcnRzLCBwZXIgbW9kZWwvY29uZmlnOgotIHNlcV9hY2NfcmF3ICAgICAgICA6IGV4YWN0IDctY2hhciBtYXRjaCwgYXJnbWF4IG9ubHkKLSBzZXFfYWNjX3Bvc3QgICAgICAgOiBleGFjdCBtYXRjaCBhZnRlciBydWxlLWJhc2VkIHBvc3QtcHJvY2Vzc2luZwogICAgICAgICAgICAgICAgICAgICAgIChsYXlvdXQtYXdhcmUgd2hlbiB0aGUgbW9kZWwgaGFzIGEgbGF5b3V0IGhlYWQpCi0gY2hhcl9hY2MgICAgICAgICAgIDogcGVyLWNoYXJhY3RlciBhY2N1cmFjeQotIGxheW91dF9hY2MgICAgICAgICA6IGxheW91dC1jbGFzc2lmaWNhdGlvbiBhY2N1cmFjeSAoaWYgaGVhZCBwcmVzZW50KQoKYGBydW5fY29tcGFyaXNvbmBgIGJ1aWxkcyBzZXZlcmFsIGNvbmZpZ3MsIG9wdGlvbmFsbHkgbG9hZHMgY2hlY2twb2ludHMsIGFuZApyZXR1cm5zIGEgdGFibGUgc28gYmFzZWxpbmVzIC8gZnVzaW9ucyAvIFNSIHZhcmlhbnRzIGNhbiBiZSBjb21wYXJlZCBoZWFkLXRvLWhlYWQuCiIiIgoKaW1wb3J0IHRvcmNoCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCgpmcm9tIHV0aWxzIGltcG9ydCBwcmVkaWN0aW9uX2Zyb21fbG9naXRzLCBsb2FkX2NoZWNrcG9pbnQKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBldmFsdWF0ZV9tb2RlbChtb2RlbCwgbG9hZGVyLCBjZmcsIGRldmljZT0iY3B1Iik6CiAgICBtb2RlbC5ldmFsKCkKICAgIHZvY2FiID0gY2ZnLnZvY2FiCiAgICBsYXlvdXRzID0gbGlzdChjZmcubGF5b3V0cykKCiAgICBuID0gMAogICAgc2VxX3JhdyA9IHNlcV9wb3N0ID0gY2hhcl9jb3JyZWN0ID0gY2hhcl90b3RhbCA9IGxheW91dF9jb3JyZWN0ID0gbGF5b3V0X3RvdGFsID0gMAoKICAgIGZvciBiYXRjaCBpbiB0cWRtKGxvYWRlciwgZGVzYz0iRXZhbCIsIGxlYXZlPUZhbHNlKToKICAgICAgICBpbWFnZXMgPSBiYXRjaFsiaW1hZ2VzIl0udG8oZGV2aWNlKQogICAgICAgIHRhcmdldHMgPSBiYXRjaFsidGFyZ2V0cyJdCiAgICAgICAgZ3RzID0gWyIiLmpvaW4odm9jYWJbaV0gZm9yIGkgaW4gdC50b2xpc3QoKSkgZm9yIHQgaW4gdGFyZ2V0c10KCiAgICAgICAgb3V0ID0gbW9kZWwoaW1hZ2VzLCByZXR1cm5fYXV4PVRydWUpCiAgICAgICAgbG9naXRzID0gb3V0WyJsb2dpdHMiXS5jcHUoKQoKICAgICAgICByYXcgPSBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKGxvZ2l0cywgdm9jYWIsIGFwcGx5X3Bvc3Rwcm9jZXNzPUZhbHNlKQogICAgICAgIGlmIGNmZy51c2VfbGF5b3V0X2hlYWQgYW5kIG91dFsibGF5b3V0Il0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHBvc3QgPSBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKGxvZ2l0cywgdm9jYWIsIGxheW91dHM9bGF5b3V0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbGF5b3V0X2xvZ2l0cz1vdXRbImxheW91dCJdLmNwdSgpLCBhcHBseV9wb3N0cHJvY2Vzcz1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHBvc3QgPSBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKGxvZ2l0cywgdm9jYWIsIGFwcGx5X3Bvc3Rwcm9jZXNzPVRydWUpCgogICAgICAgIGZvciByLCBwLCBnIGluIHppcChyYXcsIHBvc3QsIGd0cyk6CiAgICAgICAgICAgIHNlcV9yYXcgKz0gaW50KHIgPT0gZykKICAgICAgICAgICAgc2VxX3Bvc3QgKz0gaW50KHAgPT0gZykKICAgICAgICAgICAgY2hhcl9jb3JyZWN0ICs9IHN1bShhID09IGIgZm9yIGEsIGIgaW4gemlwKHAsIGcpKQogICAgICAgICAgICBjaGFyX3RvdGFsICs9IGxlbihnKQogICAgICAgIG4gKz0gbGVuKGd0cykKCiAgICAgICAgaWYgY2ZnLnVzZV9sYXlvdXRfaGVhZCBhbmQgb3V0WyJsYXlvdXQiXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgbGFiID0gYmF0Y2hbImxheW91dCJdCiAgICAgICAgICAgIG1hc2sgPSBsYWIgPj0gMAogICAgICAgICAgICBpZiBtYXNrLmFueSgpOgogICAgICAgICAgICAgICAgcHJlZF9sYXlvdXQgPSBvdXRbImxheW91dCJdLmNwdSgpLmFyZ21heChkaW09MSkKICAgICAgICAgICAgICAgIGxheW91dF9jb3JyZWN0ICs9IChwcmVkX2xheW91dFttYXNrXSA9PSBsYWJbbWFza10pLnN1bSgpLml0ZW0oKQogICAgICAgICAgICAgICAgbGF5b3V0X3RvdGFsICs9IGludChtYXNrLnN1bSgpKQoKICAgIHJldHVybiB7CiAgICAgICAgInNlcV9hY2NfcmF3Ijogc2VxX3JhdyAvIG4gaWYgbiBlbHNlIDAuMCwKICAgICAgICAic2VxX2FjY19wb3N0Ijogc2VxX3Bvc3QgLyBuIGlmIG4gZWxzZSAwLjAsCiAgICAgICAgImNoYXJfYWNjIjogY2hhcl9jb3JyZWN0IC8gY2hhcl90b3RhbCBpZiBjaGFyX3RvdGFsIGVsc2UgMC4wLAogICAgICAgICJsYXlvdXRfYWNjIjogKGxheW91dF9jb3JyZWN0IC8gbGF5b3V0X3RvdGFsKSBpZiBsYXlvdXRfdG90YWwgZWxzZSBOb25lLAogICAgICAgICJuIjogbiwKICAgIH0KCgpkZWYgZm9ybWF0X3RhYmxlKHJvd3MpOgogICAgIiIicm93czogbGlzdCBvZiBkaWN0cyB3aXRoICduYW1lJyArIG1ldHJpYyBrZXlzIC0+IG1hcmtkb3duIHRhYmxlIHN0cmluZy4iIiIKICAgIGNvbHMgPSBbIm5hbWUiLCAic2VxX2FjY19yYXciLCAic2VxX2FjY19wb3N0IiwgImNoYXJfYWNjIiwgImxheW91dF9hY2MiLCAibiJdCiAgICBoZWFkZXIgPSAifCAiICsgIiB8ICIuam9pbihjb2xzKSArICIgfCIKICAgIHNlcCA9ICJ8ICIgKyAiIHwgIi5qb2luKFsiLS0tIl0gKiBsZW4oY29scykpICsgIiB8IgogICAgbGluZXMgPSBbaGVhZGVyLCBzZXBdCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIHZhbHMgPSBbXQogICAgICAgIGZvciBjIGluIGNvbHM6CiAgICAgICAgICAgIHYgPSByLmdldChjKQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHYsIGZsb2F0KToKICAgICAgICAgICAgICAgIHZhbHMuYXBwZW5kKGYie3Y6LjRmfSIpCiAgICAgICAgICAgIGVsaWYgdiBpcyBOb25lOgogICAgICAgICAgICAgICAgdmFscy5hcHBlbmQoIi0iKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgdmFscy5hcHBlbmQoc3RyKHYpKQogICAgICAgIGxpbmVzLmFwcGVuZCgifCAiICsgIiB8ICIuam9pbih2YWxzKSArICIgfCIpCiAgICByZXR1cm4gIlxuIi5qb2luKGxpbmVzKQoKCmRlZiBydW5fY29tcGFyaXNvbihjb25maWdzLCBsb2FkZXJfYnVpbGRlciwgZGV2aWNlPSJjcHUiKToKICAgICIiIkV2YWx1YXRlIHNldmVyYWwgY29uZmlncyBhbmQgcmV0dXJuIGNvbXBhcmlzb24gcm93cy4KCiAgICBBcmdzOgogICAgICAgIGNvbmZpZ3M6IGxpc3Qgb2YgKG5hbWUsIGNmZywgY2hlY2twb2ludF9wYXRoX29yX05vbmUpLgogICAgICAgIGxvYWRlcl9idWlsZGVyOiBjYWxsYWJsZShjZmcpIC0+IERhdGFMb2FkZXIgb3ZlciBgYHRlc3RfbGFiZWxgYCB1c2luZwogICAgICAgICAgICBgYGNvbGxhdGVfaGFybmVzc2BgIChidWlsdCBwZXItY2ZnIHNvIFNSL2xheW91dCB0YXJnZXRzIG1hdGNoKS4KICAgICIiIgogICAgZnJvbSBtb2RlbHMgaW1wb3J0IGJ1aWxkX21vZGVsCgogICAgcm93cyA9IFtdCiAgICBmb3IgbmFtZSwgY2ZnLCBja3B0IGluIGNvbmZpZ3M6CiAgICAgICAgbW9kZWwgPSBidWlsZF9tb2RlbChjZmcpLnRvKGRldmljZSkKICAgICAgICBpZiBja3B0IGlzIG5vdCBOb25lOgogICAgICAgICAgICBtaXNzaW5nLCB1bmV4cGVjdGVkID0gbG9hZF9jaGVja3BvaW50KG1vZGVsLCBja3B0LCBtYXBfbG9jYXRpb249ZGV2aWNlLCBzdHJpY3Q9RmFsc2UpCiAgICAgICAgICAgIHByaW50KGYiW3tuYW1lfV0gbG9hZGVkIHtja3B0fSB8IG1pc3Npbmc9e2xlbihtaXNzaW5nKX0gdW5leHBlY3RlZD17bGVuKHVuZXhwZWN0ZWQpfSIpCiAgICAgICAgbG9hZGVyID0gbG9hZGVyX2J1aWxkZXIoY2ZnKQogICAgICAgIG1ldHJpY3MgPSBldmFsdWF0ZV9tb2RlbChtb2RlbCwgbG9hZGVyLCBjZmcsIGRldmljZT1kZXZpY2UpCiAgICAgICAgbWV0cmljc1sibmFtZSJdID0gbmFtZQogICAgICAgIHJvd3MuYXBwZW5kKG1ldHJpY3MpCiAgICAgICAgcHJpbnQoZiJbe25hbWV9XSB7bWV0cmljc30iKQogICAgcmV0dXJuIHJvd3MK",
"predict/__init__.py": "ZnJvbSAucHJlZGljdG9yIGltcG9ydCBwcmVkaWN0X2JsaW5kX3Rlc3QNCg0KX19hbGxfXyA9IFsicHJlZGljdF9ibGluZF90ZXN0Il0NCg==",
"predict/predictor.py": "aW1wb3J0IG9zDQoNCmltcG9ydCB0b3JjaA0KZnJvbSB0cWRtLmF1dG8gaW1wb3J0IHRxZG0NCg0KZnJvbSB1dGlscyBpbXBvcnQgZGVjb2RlX3ByZWQsIHByZWRpY3Rpb25fZnJvbV9sb2dpdHMNCg0KDQpkZWYgcHJlZGljdF9ibGluZF90ZXN0KG1vZGVsLCB0ZXN0X2xvYWRlciwgdm9jYWI6IHN0ciwgZGV2aWNlPSJjcHUiLCBzYXZlX3BhdGg9Ii4vdGVzdF9wcmVkaWN0aW9ucy5jc3YiLA0KICAgICAgICAgICAgICAgICAgICAgICBjaGVja3BvaW50X3BhdGg9Tm9uZSwgYXBwbHlfcG9zdHByb2Nlc3M9RmFsc2UsIGxheW91dHM9Tm9uZSk6DQogICAgaWYgY2hlY2twb2ludF9wYXRoIGlzIG5vdCBOb25lOg0KICAgICAgICBpZiBub3Qgb3MucGF0aC5pc2ZpbGUoY2hlY2twb2ludF9wYXRoKToNCiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ2hlY2twb2ludCBub3QgZm91bmQ6IHtjaGVja3BvaW50X3BhdGh9IikNCg0KICAgICAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChjaGVja3BvaW50X3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UpDQogICAgICAgIHN0YXRlX2RpY3QgPSBjaGVja3BvaW50LmdldCgibW9kZWxfc3RhdGVfZGljdCIsIGNoZWNrcG9pbnQpDQoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlX2RpY3QpDQogICAgICAgIGV4Y2VwdCBSdW50aW1lRXJyb3I6DQogICAgICAgICAgICBpZiBhbnkoay5zdGFydHN3aXRoKCJtb2R1bGUuIikgZm9yIGsgaW4gc3RhdGVfZGljdC5rZXlzKCkpOg0KICAgICAgICAgICAgICAgIHN0YXRlX2RpY3QgPSB7ay5yZXBsYWNlKCJtb2R1bGUuIiwgIiIsIDEpOiB2IGZvciBrLCB2IGluIHN0YXRlX2RpY3QuaXRlbXMoKX0NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgc3RhdGVfZGljdCA9IHtmIm1vZHVsZS57a30iOiB2IGZvciBrLCB2IGluIHN0YXRlX2RpY3QuaXRlbXMoKX0NCiAgICAgICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0KQ0KDQogICAgICAgIHByaW50KGYiTG9hZGVkIG1vZGVsIHdlaWdodHMgZnJvbToge2NoZWNrcG9pbnRfcGF0aH0iKQ0KDQogICAgbW9kZWwuZXZhbCgpDQogICAgcmVzdWx0cyA9IFtdDQoNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgdXNlX2xheW91dCA9IGFwcGx5X3Bvc3Rwcm9jZXNzIGFuZCBnZXRhdHRyKG1vZGVsLCAidXNlX2xheW91dF9oZWFkIiwgRmFsc2UpDQogICAgICAgIGZvciBpbWFnZXMsIHRyYWNrX2lkcyBpbiB0cWRtKHRlc3RfbG9hZGVyLCBkZXNjPSJQcmVkaWN0IiwgbGVhdmU9RmFsc2UpOg0KICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSkNCiAgICAgICAgICAgIGlmIHVzZV9sYXlvdXQ6DQogICAgICAgICAgICAgICAgb3V0ID0gbW9kZWwoaW1hZ2VzLCByZXR1cm5fYXV4PVRydWUpDQogICAgICAgICAgICAgICAgcHJlZHMgPSBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKA0KICAgICAgICAgICAgICAgICAgICBvdXRbImxvZ2l0cyJdLmNwdSgpLCB2b2NhYiwgbGF5b3V0cz1sYXlvdXRzLA0KICAgICAgICAgICAgICAgICAgICBsYXlvdXRfbG9naXRzPW91dFsibGF5b3V0Il0uY3B1KCksIGFwcGx5X3Bvc3Rwcm9jZXNzPVRydWUsDQogICAgICAgICAgICAgICAgKQ0KICAgICAgICAgICAgZWxpZiBhcHBseV9wb3N0cHJvY2VzczoNCiAgICAgICAgICAgICAgICBsb2dpdHMgPSBtb2RlbChpbWFnZXMpDQogICAgICAgICAgICAgICAgcHJlZHMgPSBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKGxvZ2l0cy5jcHUoKSwgdm9jYWIsIGFwcGx5X3Bvc3Rwcm9jZXNzPVRydWUpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGltYWdlcykNCiAgICAgICAgICAgICAgICBwcmVkcyA9IGRlY29kZV9wcmVkKGxvZ2l0cy5jcHUoKSwgdm9jYWIpDQoNCiAgICAgICAgICAgIGZvciB0cmFja19pZCwgcHJlZCBpbiB6aXAodHJhY2tfaWRzLCBwcmVkcyk6DQogICAgICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoeyJ0cmFja19pZCI6IHRyYWNrX2lkLCAicGxhdGVfdGV4dCI6IHByZWR9KQ0KDQogICAgZGVmIF9zb3J0X2tleShyb3cpOg0KICAgICAgICB0cmFja19pZCA9IHJvd1sidHJhY2tfaWQiXQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICByZXR1cm4gaW50KHRyYWNrX2lkLnNwbGl0KCJfIilbLTFdKQ0KICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoNCiAgICAgICAgICAgIHJldHVybiB0cmFja19pZA0KDQogICAgcmVzdWx0cy5zb3J0KGtleT1fc29ydF9rZXkpDQoNCiAgICB3aXRoIG9wZW4oc2F2ZV9wYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6DQogICAgICAgIGYud3JpdGUoInRyYWNrX2lkLHBsYXRlX3RleHRcbiIpDQogICAgICAgIGZvciByb3cgaW4gcmVzdWx0czoNCiAgICAgICAgICAgIGYud3JpdGUoZiJ7cm93Wyd0cmFja19pZCddfSx7cm93WydwbGF0ZV90ZXh0J119XG4iKQ0KDQogICAgcHJpbnQoZiJTYXZlZCB7bGVuKHJlc3VsdHMpfSBwcmVkaWN0aW9ucyB0bzoge3NhdmVfcGF0aH0iKQ0KICAgIHJldHVybiByZXN1bHRzDQo=",
"utils/__init__.py": "ZnJvbSAudGV4dF9jb2RlYyBpbXBvcnQgZGVjb2RlX3ByZWQsIGVuY29kZV9sYWJlbA0KZnJvbSAucG9zdHByb2Nlc3MgaW1wb3J0IHBvc3Rwcm9jZXNzX3J1bGVfYmFzZSwgcHJlZGljdGlvbl9mcm9tX2xvZ2l0cw0KZnJvbSAuY2hlY2twb2ludCBpbXBvcnQgbG9hZF9jaGVja3BvaW50LCByZW1hcF9sZWdhY3lfc3RhdGVfZGljdA0KDQpfX2FsbF9fID0gWw0KICAgICJlbmNvZGVfbGFiZWwiLA0KICAgICJkZWNvZGVfcHJlZCIsDQogICAgInBvc3Rwcm9jZXNzX3J1bGVfYmFzZSIsDQogICAgInByZWRpY3Rpb25fZnJvbV9sb2dpdHMiLA0KICAgICJsb2FkX2NoZWNrcG9pbnQiLA0KICAgICJyZW1hcF9sZWdhY3lfc3RhdGVfZGljdCIsDQpdDQo=",
"utils/checkpoint.py": "IiIiQ2hlY2twb2ludCBsb2FkaW5nIHdpdGggYmFja3dhcmQtY29tcGF0IGtleSByZW1hcHBpbmcuCgpUaGUgcmVnaXN0cnkgbW9kZWwgcmVuYW1lcyBhIGZldyBzdWJtb2R1bGVzIHJlbGF0aXZlIHRvIHRoZSBvcmlnaW5hbCBub3RlYm9va3M6CiAgICBhdHRuX2Z1c2lvbi4qICAgICAgIC0+IGZ1c2lvbi4qCiAgICBwb3NfZW5jb2Rlci4qICAgICAgIC0+IGRlY29kZXIucG9zX2VuY29kZXIuKgogICAgdHJhbnNmb3JtZXJfbGF5ZXIuKiAtPiBkZWNvZGVyLmVuY29kZXIuKgogICAgc2VxdWVuY2VfbW9kZWwuKiAgICAgLT4gZGVjb2Rlci4qICAgICAgICAoQmlMU1RNIHZhcmlhbnQpCnNvIHRoYXQgdGhlIGxlZ2FjeSAwMi8wMyBSZXNOZXQrVHJhbnNmb3JtZXIgY2hlY2twb2ludHMgbG9hZCBpbnRvIGl0LgoiIiIKCmltcG9ydCB0b3JjaAoKX0xFR0FDWV9QUkVGSVhfUkVNQVAgPSBbCiAgICAoJ2F0dG5fZnVzaW9uLicsICdmdXNpb24uJyksCiAgICAoJ3Bvc19lbmNvZGVyLicsICdkZWNvZGVyLnBvc19lbmNvZGVyLicpLAogICAgKCd0cmFuc2Zvcm1lcl9sYXllci4nLCAnZGVjb2Rlci5lbmNvZGVyLicpLAogICAgKCdzZXF1ZW5jZV9tb2RlbC4nLCAnZGVjb2Rlci4nKSwKXQoKCmRlZiByZW1hcF9sZWdhY3lfc3RhdGVfZGljdChzdGF0ZV9kaWN0KToKICAgIG5ld19zZCA9IHt9CiAgICBmb3IgaywgdiBpbiBzdGF0ZV9kaWN0Lml0ZW1zKCk6CiAgICAgICAgbmsgPSBrCiAgICAgICAgZm9yIG9sZCwgbmV3IGluIF9MRUdBQ1lfUFJFRklYX1JFTUFQOgogICAgICAgICAgICBpZiBuay5zdGFydHN3aXRoKG9sZCk6CiAgICAgICAgICAgICAgICBuayA9IG5ldyArIG5rW2xlbihvbGQpOl0KICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgbmV3X3NkW25rXSA9IHYKICAgIHJldHVybiBuZXdfc2QKCgpkZWYgbG9hZF9jaGVja3BvaW50KG1vZGVsLCBja3B0X3BhdGgsIG1hcF9sb2NhdGlvbj0nY3B1Jywgc3RyaWN0PUZhbHNlKToKICAgICIiIkxvYWQgYSBjaGVja3BvaW50IGludG8gYGBtb2RlbGBgLCByZW1hcHBpbmcgbGVnYWN5IGtleXMuIFJldHVybnMgdGhlCiAgICAobWlzc2luZ19rZXlzLCB1bmV4cGVjdGVkX2tleXMpIHR1cGxlIGZyb20gYGBsb2FkX3N0YXRlX2RpY3RgYC4iIiIKICAgIHN0YXRlID0gdG9yY2gubG9hZChja3B0X3BhdGgsIG1hcF9sb2NhdGlvbj1tYXBfbG9jYXRpb24pCiAgICBpZiBpc2luc3RhbmNlKHN0YXRlLCBkaWN0KToKICAgICAgICBmb3Iga2V5IGluICgnbW9kZWxfc3RhdGVfZGljdCcsICdzdGF0ZV9kaWN0Jyk6CiAgICAgICAgICAgIGlmIGtleSBpbiBzdGF0ZToKICAgICAgICAgICAgICAgIHN0YXRlID0gc3RhdGVba2V5XQogICAgICAgICAgICAgICAgYnJlYWsKICAgIHN0YXRlID0gcmVtYXBfbGVnYWN5X3N0YXRlX2RpY3Qoc3RhdGUpCiAgICByZXR1cm4gbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlLCBzdHJpY3Q9c3RyaWN0KQo=",
"utils/postprocess.py": "IiIiUnVsZS1iYXNlZCBwb3N0LXByb2Nlc3NpbmcgZm9yIEJyYXppbGlhbiAvIE1lcmNvc3VyIHBsYXRlcy4KClBvc2l0aW9uIGdyYW1tYXIgKHZhbGlkYXRlZCBvbiB0aGUgZGF0YXNldCk6CiAgICBpZHggMCwxLDIgLT4gbGV0dGVyIChhbGwgbGF5b3V0cykKICAgIGlkeCAzICAgICAtPiBkaWdpdCAgKGFsbCBsYXlvdXRzKQogICAgaWR4IDQgICAgIC0+IGRpZ2l0IGZvciBCcmF6aWxpYW4gKExMTEREREQpLCBsZXR0ZXIgZm9yIE1lcmNvc3VyIChMTExETEREKQogICAgaWR4IDUsNiAgIC0+IGRpZ2l0ICAoYWxsIGxheW91dHMpCgpUaGUgb3JpZ2luYWwgbm90ZWJvb2sgcnVsZSB3YXMgbGF5b3V0LWFnbm9zdGljIGFuZCBmb3JjZWQgaWR4IDQgdG93YXJkIGEgZGlnaXQsCndoaWNoIGlzIHdyb25nIGZvciBNZXJjb3N1ci4gUGFzc2luZyBgYGxheW91dGBgIG1ha2VzIGlkeCA0IGNvcnJlY3Q7IHdoZW4gdGhlCmxheW91dCBpcyB1bmtub3duIChibGluZCB0ZXN0IHdpdGhvdXQgYSBsYXlvdXQgaGVhZCkgaWR4IDQgaXMgbGVmdCB1bnRvdWNoZWQuCiIiIgoKIyBkaWdpdCAtPiB2aXN1YWxseS1zaW1pbGFyIGxldHRlcgpEMkMgPSB7JzAnOiAnTycsICcxJzogJ0knLCAnMic6ICdaJywgJzUnOiAnUycsICc2JzogJ0cnLCAnOCc6ICdCJywgJzQnOiAnQSd9CiMgbGV0dGVyIC0+IHZpc3VhbGx5LXNpbWlsYXIgZGlnaXQKQzJEID0ge3Y6IGsgZm9yIGssIHYgaW4gRDJDLml0ZW1zKCl9CgoKZGVmIF90b19sZXR0ZXIoY2gpOgogICAgcmV0dXJuIEQyQy5nZXQoY2gsIGNoKQoKCmRlZiBfdG9fZGlnaXQoY2gpOgogICAgcmV0dXJuIEMyRC5nZXQoY2gsIGNoKQoKCmRlZiBwb3N0cHJvY2Vzc19ydWxlX2Jhc2UocHJlZDogc3RyLCBsYXlvdXQ6IHN0ciA9IE5vbmUpIC0+IHN0cjoKICAgICIiIkNvZXJjZSBhIDctY2hhciBwcmVkaWN0aW9uIHRvIHRoZSBwbGF0ZSBncmFtbWFyLiBgYGxheW91dGBgIGlzIG9wdGlvbmFsLiIiIgogICAgY2hhcnMgPSBsaXN0KHByZWQudXBwZXIoKSkKICAgIGlmIGxlbihjaGFycykgIT0gNzoKICAgICAgICByZXR1cm4gcHJlZC51cHBlcigpCgogICAgZm9yIGkgaW4gKDAsIDEsIDIpOgogICAgICAgIGNoYXJzW2ldID0gX3RvX2xldHRlcihjaGFyc1tpXSkKICAgIGNoYXJzWzNdID0gX3RvX2RpZ2l0KGNoYXJzWzNdKQoKICAgIGlmIGxheW91dCA9PSAnQnJhemlsaWFuJzoKICAgICAgICBjaGFyc1s0XSA9IF90b19kaWdpdChjaGFyc1s0XSkKICAgIGVsaWYgbGF5b3V0ID09ICdNZXJjb3N1cic6CiAgICAgICAgY2hhcnNbNF0gPSBfdG9fbGV0dGVyKGNoYXJzWzRdKQogICAgZWxzZToKICAgICAgICAjIFVua25vd24gbGF5b3V0OiBrZWVwIHRoZSBsZWdhY3kgc29mdCBmaXggKE8gLT4gMCkgb25seS4KICAgICAgICBpZiBjaGFyc1s0XSA9PSAnTyc6CiAgICAgICAgICAgIGNoYXJzWzRdID0gJzAnCgogICAgZm9yIGkgaW4gKDUsIDYpOgogICAgICAgIGNoYXJzW2ldID0gX3RvX2RpZ2l0KGNoYXJzW2ldKQoKICAgIHJldHVybiAnJy5qb2luKGNoYXJzKQoKCmRlZiBwcmVkaWN0aW9uX2Zyb21fbG9naXRzKGxvZ2l0cywgdm9jYWIsIGxheW91dHM9Tm9uZSwgbGF5b3V0X2xvZ2l0cz1Ob25lLCBhcHBseV9wb3N0cHJvY2Vzcz1UcnVlKToKICAgICIiIkRlY29kZSBsb2dpdHMgdG8gc3RyaW5ncywgb3B0aW9uYWxseSBhcHBseWluZyBsYXlvdXQtYXdhcmUgcG9zdC1wcm9jZXNzaW5nLgoKICAgIGBgbGF5b3V0X2xvZ2l0c2BgIChCLCBudW1fbGF5b3V0cykgKyBgYGxheW91dHNgYCAobGlzdCBvZiBuYW1lcykgZW5hYmxlIHRoZQogICAgbGF5b3V0LWF3YXJlIGlkeC00IHJ1bGU7IG90aGVyd2lzZSBwb3N0LXByb2Nlc3NpbmcgaXMgbGF5b3V0LWFnbm9zdGljLgogICAgIiIiCiAgICBpbmRpY2VzID0gbG9naXRzLmFyZ21heChkaW09MikuY3B1KCkKICAgIHJhdyA9IFsnJy5qb2luKHZvY2FiW2ldIGZvciBpIGluIHNlcS50b2xpc3QoKSkgZm9yIHNlcSBpbiBpbmRpY2VzXQogICAgaWYgbm90IGFwcGx5X3Bvc3Rwcm9jZXNzOgogICAgICAgIHJldHVybiByYXcKCiAgICBsYXlvdXRfbmFtZXMgPSBbTm9uZV0gKiBsZW4ocmF3KQogICAgaWYgbGF5b3V0cyBpcyBub3QgTm9uZSBhbmQgbGF5b3V0X2xvZ2l0cyBpcyBub3QgTm9uZToKICAgICAgICBwcmVkX2xheW91dCA9IGxheW91dF9sb2dpdHMuYXJnbWF4KGRpbT0xKS5jcHUoKS50b2xpc3QoKQogICAgICAgIGxheW91dF9uYW1lcyA9IFtsYXlvdXRzW2ldIGZvciBpIGluIHByZWRfbGF5b3V0XQoKICAgIHJldHVybiBbcG9zdHByb2Nlc3NfcnVsZV9iYXNlKHAsIGxheW91dD1sbikgZm9yIHAsIGxuIGluIHppcChyYXcsIGxheW91dF9uYW1lcyldCg==",
"utils/text_codec.py": "aW1wb3J0IHRvcmNoDQoNCg0KZGVmIGVuY29kZV9sYWJlbChsYWJlbDogc3RyLCB2b2NhYjogc3RyKSAtPiB0b3JjaC5UZW5zb3I6DQogICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbdm9jYWIuaW5kZXgoYykgZm9yIGMgaW4gbGFiZWwudXBwZXIoKV0sIGR0eXBlPXRvcmNoLmxvbmcpDQoNCg0KZGVmIGRlY29kZV9wcmVkKGxvZ2l0czogdG9yY2guVGVuc29yLCB2b2NhYjogc3RyKToNCiAgICBpbmRpY2VzID0gbG9naXRzLmFyZ21heChkaW09MikNCiAgICByZXR1cm4gWyIiLmpvaW4odm9jYWJbaV0gZm9yIGkgaW4gc2VxLnRvbGlzdCgpKSBmb3Igc2VxIGluIGluZGljZXNdDQo=",
"visualization/__init__.py": "ZnJvbSAudmlzdWFsaXplciBpbXBvcnQgdW5ub3JtYWxpemUsIHZpc3VhbGl6ZV9wcmVkaWN0aW9ucywgdmlzdWFsaXplX3ZhbF9zYW1wbGVzDQoNCl9fYWxsX18gPSBbInVubm9ybWFsaXplIiwgInZpc3VhbGl6ZV92YWxfc2FtcGxlcyIsICJ2aXN1YWxpemVfcHJlZGljdGlvbnMiXQ0K",
"visualization/visualizer.py": "aW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdA0KaW1wb3J0IHRvcmNoDQoNCmZyb20gdXRpbHMgaW1wb3J0IGRlY29kZV9wcmVkDQoNCg0KZGVmIHVubm9ybWFsaXplKHRlbnNvcjogdG9yY2guVGVuc29yKToNCiAgICAjIE11c3QgbWF0Y2ggdHJhaW5pbmcgbm9ybWFsaXphdGlvbiAobWVhbj1zdGQ9MC41KSwgbm90IEltYWdlTmV0IHN0YXRzLg0KICAgIG1lYW4gPSB0b3JjaC50ZW5zb3IoWzAuNSwgMC41LCAwLjVdKS52aWV3KDMsIDEsIDEpDQogICAgc3RkID0gdG9yY2gudGVuc29yKFswLjUsIDAuNSwgMC41XSkudmlldygzLCAxLCAxKQ0KICAgIHJldHVybiAodGVuc29yICogc3RkICsgbWVhbikuY2xhbXAoMCwgMSkucGVybXV0ZSgxLCAyLCAwKS5udW1weSgpDQoNCg0KZGVmIHZpc3VhbGl6ZV92YWxfc2FtcGxlcyhtb2RlbCwgdmFsX2xvYWRlciwgdm9jYWI6IHN0ciwgZGV2aWNlOiBzdHIgPSAiY3B1Iiwgbl9zYW1wbGVzOiBpbnQgPSAyKToNCiAgICBtb2RlbC5ldmFsKCkNCiAgICBpbWFnZXMsIHRhcmdldHMgPSBuZXh0KGl0ZXIodmFsX2xvYWRlcikpDQogICAgaW1hZ2VzX2RldiA9IGltYWdlc1s6bl9zYW1wbGVzXS50byhkZXZpY2UpDQoNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzX2RldikNCg0KICAgIHByZWRzID0gZGVjb2RlX3ByZWQobG9naXRzLmNwdSgpLCB2b2NhYikNCiAgICBndHMgPSBbIiIuam9pbih2b2NhYltpXSBmb3IgaSBpbiB0LnRvbGlzdCgpKSBmb3IgdCBpbiB0YXJnZXRzWzpuX3NhbXBsZXNdXQ0KDQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKG5fc2FtcGxlcywgMSwgZmlnc2l6ZT0oNiwgMi41ICogbl9zYW1wbGVzKSkNCiAgICBpZiBuX3NhbXBsZXMgPT0gMToNCiAgICAgICAgYXhlcyA9IFtheGVzXQ0KDQogICAgZm9yIGkgaW4gcmFuZ2Uobl9zYW1wbGVzKToNCiAgICAgICAgYXggPSBheGVzW2ldDQogICAgICAgIGltZyA9IHVubm9ybWFsaXplKGltYWdlc1tpLCAwXS5jcHUoKSkNCiAgICAgICAgYXguaW1zaG93KGltZykNCiAgICAgICAgYXguYXhpcygib2ZmIikNCg0KICAgICAgICBjb3JyZWN0ID0gcHJlZHNbaV0gPT0gZ3RzW2ldDQogICAgICAgIGNvbG9yID0gImdyZWVuIiBpZiBjb3JyZWN0IGVsc2UgInJlZCINCiAgICAgICAgYXguc2V0X3RpdGxlKGYiR1Q6IHtndHNbaV19ICAgfCAgIFByZWQ6IHtwcmVkc1tpXX0iLCBmb250c2l6ZT0xMSwgY29sb3I9Y29sb3IsIGZvbnR3ZWlnaHQ9ImJvbGQiKQ0KDQogICAgcGx0LnN1cHRpdGxlKCJWYWwgc2FtcGxlIHByZXZpZXciLCBmb250c2l6ZT0xMiwgeT0xLjAxKQ0KICAgIHBsdC50aWdodF9sYXlvdXQoKQ0KICAgIHBsdC5zaG93KCkNCg0KDQpkZWYgdmlzdWFsaXplX3ByZWRpY3Rpb25zKG1vZGVsLCBsb2FkZXIsIHZvY2FiOiBzdHIsIGRldmljZTogc3RyID0gImNwdSIsIG5fc2FtcGxlczogaW50ID0gNiwgbW9kZTogc3RyID0gInRlc3QiKToNCiAgICBtb2RlbC5ldmFsKCkNCiAgICBiYXRjaCA9IG5leHQoaXRlcihsb2FkZXIpKQ0KICAgIGlmIG1vZGUgPT0gInRlc3QiOg0KICAgICAgICBpbWFnZXMsIHRhcmdldHMsIHRyYWNrX2lkcyA9IGJhdGNoDQogICAgZWxzZToNCiAgICAgICAgaW1hZ2VzLCB0cmFja19pZHMgPSBiYXRjaA0KDQogICAgaW1hZ2VzID0gaW1hZ2VzLnRvKGRldmljZSkNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2VzKQ0KICAgICAgICBwcmVkcyA9IGRlY29kZV9wcmVkKGxvZ2l0cy5jcHUoKSwgdm9jYWIpDQoNCiAgICBpbWFnZXNfY3B1ID0gaW1hZ2VzLmRldGFjaCgpLmNwdSgpDQogICAgbl9zYW1wbGVzID0gbWluKG5fc2FtcGxlcywgaW1hZ2VzX2NwdS5zaXplKDApKQ0KICAgIG5fZnJhbWVzID0gaW1hZ2VzX2NwdS5zaXplKDEpDQoNCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMobl9zYW1wbGVzLCBuX2ZyYW1lcywgZmlnc2l6ZT0obl9mcmFtZXMgKiAzLCBuX3NhbXBsZXMgKiAyLjUpKQ0KICAgIGlmIG5fc2FtcGxlcyA9PSAxOg0KICAgICAgICBheGVzID0gW2F4ZXNdDQoNCiAgICBmb3Igcm93X2lkeCBpbiByYW5nZShuX3NhbXBsZXMpOg0KICAgICAgICBwcmVkX2xhYmVsID0gcHJlZHNbcm93X2lkeF0NCiAgICAgICAgdHJhY2tfaWQgPSB0cmFja19pZHNbcm93X2lkeF0NCg0KICAgICAgICBpZiBtb2RlID09ICJ0ZXN0IjoNCiAgICAgICAgICAgIGd0ID0gIiIuam9pbih2b2NhYltpXSBmb3IgaSBpbiB0YXJnZXRzW3Jvd19pZHhdLnRvbGlzdCgpKQ0KICAgICAgICAgICAgY29ycmVjdCA9IHByZWRfbGFiZWwgPT0gZ3QNCiAgICAgICAgICAgIGNvbG9yID0gIiM0Q0FGNTAiIGlmIGNvcnJlY3QgZWxzZSAiI0Y0NDMzNiINCiAgICAgICAgICAgIHRpdGxlID0gZiJ7dHJhY2tfaWR9XG5HVDoge2d0fSAgfCAgUHJlZDoge3ByZWRfbGFiZWx9Ig0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgY29sb3IgPSAiIzIxOTZGMyINCiAgICAgICAgICAgIHRpdGxlID0gZiJ7dHJhY2tfaWR9XG5QcmVkOiB7cHJlZF9sYWJlbH0iDQoNCiAgICAgICAgZm9yIGNvbF9pZHggaW4gcmFuZ2Uobl9mcmFtZXMpOg0KICAgICAgICAgICAgYXggPSBheGVzW3Jvd19pZHhdW2NvbF9pZHhdDQogICAgICAgICAgICBpbWcgPSB1bm5vcm1hbGl6ZShpbWFnZXNfY3B1W3Jvd19pZHgsIGNvbF9pZHhdKQ0KICAgICAgICAgICAgYXguaW1zaG93KGltZykNCiAgICAgICAgICAgIGF4LmF4aXMoIm9mZiIpDQogICAgICAgICAgICBheC5zZXRfeGxhYmVsKGYiZnJhbWUge2NvbF9pZHggKyAxfSIsIGZvbnRzaXplPTcsIGxhYmVscGFkPTIpDQoNCiAgICAgICAgICAgIGlmIGNvbF9pZHggPT0gMDoNCiAgICAgICAgICAgICAgICBheC5zZXRfdGl0bGUodGl0bGUsIGZvbnRzaXplPTksIGZvbnR3ZWlnaHQ9ImJvbGQiLCBjb2xvcj1jb2xvciwgbG9jPSJsZWZ0IiwgcGFkPTMpDQoNCiAgICBwbHQuc3VwdGl0bGUoZid7IlRlc3QiIGlmIG1vZGUgPT0gInRlc3QiIGVsc2UgIkJsaW5kIFRlc3QifSAtIEluZmVyZW5jZSBQcmV2aWV3JywgZm9udHNpemU9MTIsIGZvbnR3ZWlnaHQ9ImJvbGQiLCB5PTEuMDEpDQogICAgcGx0LnRpZ2h0X2xheW91dCgpDQogICAgcGx0LnNhdmVmaWcoZiJpbmZlcmVuY2VfcHJldmlld197bW9kZX0ucG5nIiwgZHBpPTEyMCwgYmJveF9pbmNoZXM9InRpZ2h0IikNCiAgICBwbHQuc2hvdygpDQo="
}

for _rel, _b64 in _FILES.items():
    _p = os.path.join(SRC_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, 'wb') as _f:
        _f.write(base64.b64decode(_b64))

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print('wrote', len(_FILES), 'code files to', SRC_DIR)


In [ ]:
from functools import partial
import torch
from torch.utils.data import DataLoader

from config import make_config
from models import build_model
from datasets import (ICPRDataSet, build_transforms, build_sr_target_transform,
                      collate_harness, collate_fn_blind_test)
from trainers.harness import train_harness
from evaluation import evaluate_model, format_table
from predict import predict_blind_test
from utils import load_checkpoint


## 2. Experiments — A = weight 03 (eval-only), C = temporal_transformer (no SR)

In [ ]:
BACKBONE = 'resnet34'
FUSION   = 'temporal_transformer'   # mean | max | attention | frame_quality | temporal_transformer

COMMON = dict(
    path=DATA_ROOT,
    batch_size=64,
    lr=5e-4,
    epochs=50,                # match the 03 recipe
    warmup_epochs=3,
    early_stop_count=10,
    seed=42,
    extractor_pretrained=True,
    freeze_extractor=False,
    use_amp=True,             # mixed precision on GPU (no-op on CPU)
)

# (name, mode, overrides) -- mode: 'pretrained' = load weight 03 & eval only; 'train' = train
EXPERIMENTS = [
    ('A_baseline_03', 'pretrained', dict(fusion_type='attention', use_sr=False, use_layout_head=False)),
    ('C_fusion_only', 'train',      dict(fusion_type=FUSION,       use_sr=False, use_layout_head=False)),
]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
for name, mode, ov in EXPERIMENTS:
    print(f'  {name} [{mode}]: {ov}')


## 3. Helpers

In [ ]:
def build_cfg(name, overrides):
    cfg = make_config(f'{BACKBONE}_transformer', **{**COMMON, **overrides})
    cfg.device = DEVICE
    cfg.best_model_path = os.path.join(OUT_DIR, f'{BACKBONE}_{name}.pth')
    return cfg

def build_test_loader(cfg):
    _, val_tf = build_transforms(cfg.img_H, cfg.img_W)
    collate = partial(collate_harness, vocab=cfg.vocab)
    test_ds = ICPRDataSet(cfg.path, 'test_label', transform=val_tf,
                          return_layout=cfg.use_layout_head, layouts=cfg.layouts, vocab=cfg.vocab)
    return DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)

def build_loaders(cfg):
    train_tf, val_tf = build_transforms(cfg.img_H, cfg.img_W)
    sr_tf = build_sr_target_transform(cfg.img_H, cfg.img_W, cfg.sr_scale) if cfg.use_sr else None
    collate = partial(collate_harness, vocab=cfg.vocab)
    nw = 2 if IN_KAGGLE else 0
    train_ds = ICPRDataSet(cfg.path, 'train', transform=train_tf, sr_transform=sr_tf,
                           return_sr=cfg.use_sr, return_layout=cfg.use_layout_head,
                           layouts=cfg.layouts, vocab=cfg.vocab)
    val_ds   = ICPRDataSet(cfg.path, 'val', transform=val_tf,
                           return_layout=cfg.use_layout_head, layouts=cfg.layouts, vocab=cfg.vocab)
    test_ds  = ICPRDataSet(cfg.path, 'test_label', transform=val_tf,
                           return_layout=cfg.use_layout_head, layouts=cfg.layouts, vocab=cfg.vocab)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              collate_fn=collate, num_workers=nw, pin_memory=IN_KAGGLE)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              collate_fn=collate, num_workers=nw, pin_memory=IN_KAGGLE)
    test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)
    return train_loader, val_loader, test_loader

def eval_pretrained(name, overrides, weight_path):
    cfg = build_cfg(name, overrides)
    print(f'\n===== {name} (pretrained, eval-only) =====')
    model = build_model(cfg).to(cfg.device)
    missing, unexpected = load_checkpoint(model, weight_path, map_location=cfg.device, strict=False)
    print(f'loaded {os.path.basename(weight_path)} | missing={len(missing)} unexpected={len(unexpected)}')
    metrics = evaluate_model(model, build_test_loader(cfg), cfg, device=cfg.device)
    metrics['name'] = name
    print(f'[{name}] {metrics}')
    return cfg, model, metrics

def run_experiment(name, overrides):
    cfg = build_cfg(name, overrides)
    print(f'\n===== {name} (train) =====')
    print(f'fusion={cfg.fusion_type} use_sr={cfg.use_sr} use_layout_head={cfg.use_layout_head} '
          f'epochs={cfg.epochs} amp={cfg.use_amp}')
    train_loader, val_loader, test_loader = build_loaders(cfg)
    model = build_model(cfg).to(cfg.device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'params: {n_params:.1f}M  | checkpoint -> {cfg.best_model_path}')
    model, _ = train_harness(model, cfg, train_loader, val_loader, cfg.best_model_path, device=cfg.device)
    metrics = evaluate_model(model, test_loader, cfg, device=cfg.device)
    metrics['name'] = name
    print(f'[{name}] {metrics}')
    return cfg, model, metrics


## 4. Run — eval A (weight 03), train C (temporal_transformer, no SR)

In [ ]:
results = {}
for name, mode, overrides in EXPERIMENTS:
    if mode == 'pretrained':
        results[name] = eval_pretrained(name, overrides, WEIGHT_PATH)
    else:
        results[name] = run_experiment(name, overrides)


## 5. So sánh A vs C trên `test_label`

In [ ]:
rows = [results[name][2] for name, _, _ in EXPERIMENTS]
print(format_table(rows))


## 6. Predict blind test bằng model tốt hơn -> submission CSV

In [ ]:
best_name = max(results, key=lambda k: results[k][2]['seq_acc_post'])
best_cfg, best_model, _ = results[best_name]
print('best model:', best_name)

_, val_tf = build_transforms(best_cfg.img_H, best_cfg.img_W)
blind_ds = ICPRDataSet(best_cfg.path, 'blind_test', transform=val_tf, vocab=best_cfg.vocab)
blind_loader = DataLoader(blind_ds, batch_size=best_cfg.batch_size, shuffle=False,
                          collate_fn=collate_fn_blind_test)
csv_path = os.path.join(OUT_DIR, f'submission_{best_name}.csv')
predict_blind_test(best_model, blind_loader, best_cfg.vocab, device=best_cfg.device,
                   save_path=csv_path, apply_postprocess=True, layouts=list(best_cfg.layouts))
print('submission ->', csv_path)
